# MobileI2V ONNX Converter for Google Colab

Converts all 4 MobileI2V sub-models to ONNX format:
1. **VAE Encoder** — LTX-Video VAE encoder (128-ch latent)
2. **Qwen2 Text Encoder** — Qwen2-0.5B
3. **MobileI2V DiT UNet** — Mobiledit 300M (hybrid_371.pth)
4. **Turbo-VAED Decoder** — Lightning-fast VAE decoder

Requires: GPU runtime (T4, V100, A100, L4, or L40s)
Output: ONNX models saved to `/content/mobilei2v_onnx/`


In [1]:
# @title Configure CUDA memory allocator
# This helps prevent CUDA OutOfMemory errors due to fragmentation.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF set to expandable_segments:True")

PYTORCH_CUDA_ALLOC_CONF set to expandable_segments:True


In [2]:
# @title 1. Environment Check
import sys, subprocess, os, time, platform
import torch

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

# Check GPU VRAM
if torch.cuda.is_available():
    free_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total VRAM: {free_mem:.1f} GB")
    if free_mem < 8:
        print("Low VRAM - models will use float16 but some may OOM")
    else:
        print("Sufficient VRAM")


Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:36:12) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.12.1+cu126
CUDA available: True
GPU: NVIDIA GeForce GTX 1050 Ti
CUDA version: 12.6
Total VRAM: 4.3 GB
Low VRAM - models will use float16 but some may OOM


In [3]:
# @title 2. Install Dependencies
reqs = ["numba==0.61.0",
    "diffusers>=0.30.0", "transformers>=4.44.0","numpy==2.1.0",
    "onnxruntime-gpu==1.19.0", "onnx==1.19.0", # Changed to onnxruntime-gpu 1.19.0 for CUDA 12.8 compatibility
    "huggingface_hub>=0.24.0", "einops", "timm", "accelerate>=0.31.0","onnxscript" # Pinning accelerate for numpy 2.x compatibility
]
# No explicit numpy version needed; let pip resolve (expecting numpy>=2)
for r in reqs:
    result=subprocess.run(
        [sys.executable, "-m", "pip", "install", "-v", r],capture_output=True,
        text=True,
        check=True
    )
    #print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)
print("All dependencies installed")

STDERR:   WARNING: The scripts f2py.exe and numpy-config.exe are installed in 'C:\Users\himad\AppData\Roaming\Python\Python312\Scripts' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.

STDERR: 
STDERR: 
STDERR:   WARNING: The scripts f2py.exe and numpy-config.exe are installed in 'C:\Users\himad\AppData\Roaming\Python\Python312\Scripts' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

STDERR:   WARNING: The script humanfriendly.exe is installed in 'C:\Users\himad\AppData\Roaming\Python\Python312\Scripts' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn

In [ ]:
print("--- Installed Package Versions ---")
packages = [
    "diffusers", "transformers", "onnxruntime-gpu", "onnx",
    "huggingface_hub", "einops", "timm", "accelerate", "onnxscript","numpy"
]

for pkg in packages:
    try:
        result = subprocess.run([sys.executable, "-m", "pip", "show", pkg], capture_output=True, text=True, check=True)
        version_line = next((line for line in result.stdout.splitlines() if line.startswith("Version:")), "Version: Not found")
        print(f"{pkg}: {version_line.split(' ')[1]}")
    except subprocess.CalledProcessError:
        print(f"{pkg}: Not Installed")
    except StopIteration:
        print(f"{pkg}: Version not found")

import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version (from PyTorch): {torch.version.cuda}")

In [3]:
# @title 3. Create Directories
MODEL_DIR = "/content/mobilei2v_onnx"
CACHE_DIR = "/content/model_cache"
CODE_DIR = "/content/mobilei2v_code"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CODE_DIR, exist_ok=True)
os.makedirs(os.path.join(CODE_DIR, "models"), exist_ok=True)

print(f"Output models: {MODEL_DIR}")
print(f"Model cache:   {CACHE_DIR}")
print(f"Source code:   {CODE_DIR}")


Output models: /content/mobilei2v_onnx
Model cache:   /content/model_cache
Source code:   /content/mobilei2v_code


In [4]:
%%writefile /content/mobilei2v_code/model_utils.py
"""
Shared ONNX export utilities for MobileI2V model conversion.

Provides:
  - get_device:        Select CUDA or CPU
  - download_repo:     Selective HuggingFace repo download (entire repo or by patterns)
  - export_onnx:       Generic torch.onnx.export wrapper with verification
  - verify_onnx:       Load exported model and validate expected inputs/outputs
"""

from __future__ import annotations

import logging
import os
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import onnx
import onnxruntime as ort
import torch

# Ensure UTF-8 encoding for stdout/stderr — PyTorch ONNX exporter uses emoji
# characters in verbose output, which crash on Windows cp1252 console.
if sys.stdout and hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass
if sys.stderr and hasattr(sys.stderr, 'reconfigure'):
    try:
        sys.stderr.reconfigure(encoding='utf-8')
    except Exception:
        pass

logger = logging.getLogger(__name__)

MODEL_CACHE_DIR = Path("/content/model_cache")


def get_device() -> torch.device:
    """Return CUDA device if available, otherwise CPU."""
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
        logger.info("Using CUDA device: %s", torch.cuda.get_device_name(0))
    else:
        device = torch.device("cpu")
        logger.info("CUDA not available — using CPU")
    return device


def get_torch_dtype(device: torch.device) -> torch.dtype:
    """Return float16 on CUDA, float32 on CPU."""
    return torch.float16 if device.type == "cuda" else torch.float32


def enable_cuda_optimizations(device: torch.device) -> None:
    """Enable CUDA-specific performance optimizations for ONNX export."""
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision("high")
        logger.info("CUDA optimizations enabled: cudnn.benchmark, float32_matmul_precision=high")


def _torch_version_tuple() -> tuple:
    """Return PyTorch version as (major, minor) tuple."""
    parts = torch.__version__.split("+")[0].split(".")[:2]
    return tuple(int(x) for x in parts)


def _patch_neg_transpose(onnx_model: onnx.ModelProto) -> bool:
    """Fix Transpose nodes with -1 in their perm attribute."""
    fixed = False
    for node in onnx_model.graph.node:
        if node.op_type != "Transpose":
            continue
        for attr in node.attribute:
            if attr.name != "perm" or attr.type != onnx.AttributeProto.INTS:
                continue
            perms = list(attr.ints)
            if -1 not in perms:
                continue
            rank = len(perms)
            new_perms = [rank - 1 if p == -1 else p for p in perms]
            del attr.ints[:]
            attr.ints.extend(new_perms)
            fixed = True
    return fixed


def export_onnx(
    model: torch.nn.Module,
    model_name: str,
    output_dir: os.PathLike,
    dummy_inputs: Dict[str, torch.Tensor],
    dynamic_axes: Dict[str, Dict[int, str]],
    input_names: List[str],
    output_names: List[str],
    verbose: bool = False,
) -> Path:
    """Export a PyTorch model to ONNX using the stable legacy API.

    Uses torch.onnx.export (stable JIT-based API) with dynamo=False on
    PyTorch >= 2.12 to avoid dynamo tracing issues with RoPE buffers.
    Then applies Transpose perm repair for any -1 permute indices.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    onnx_path = output_dir / f"{model_name}.onnx"

    logger.info("Exporting %s -> %s", model_name, onnx_path)

    args = tuple(dummy_inputs[name] for name in input_names)
    torch_version = _torch_version_tuple()
    use_dynamo_false = torch_version >= (2, 12)

    with torch.no_grad():
        export_kwargs = dict(
            model=model,
            args=args,
            f=str(onnx_path),
            input_names=input_names,
            output_names=output_names,
            dynamic_axes=dynamic_axes,
            opset_version=18,
            do_constant_folding=True,
            verbose=verbose,
        )
        if use_dynamo_false:
            export_kwargs["dynamo"] = False

        torch.onnx.export(**export_kwargs)

    logger.info("Export complete — verifying ONNX model ...")
    onnx_model = onnx.load(str(onnx_path))

    if _patch_neg_transpose(onnx_model):
        logger.info("Repaired Transpose node(s) with -1 perm")
        onnx.save(onnx_model, str(onnx_path))
        onnx_model = onnx.load(str(onnx_path))

    onnx.checker.check_model(onnx_model)
    logger.info("ONNX check passed for %s", onnx_path)

    file_size_mb = onnx_path.stat().st_size / (1024 * 1024)
    logger.info("%s size: %.2f MB", onnx_path.name, file_size_mb)

    return onnx_path


def verify_onnx(
    onnx_path: os.PathLike,
    feeds: Dict[str, Any],
    expected_output_names: Sequence[str],
    rtol: float = 1e-3,
    atol: float = 1e-3,
    providers: Optional[List[str]] = None,
) -> bool:
    """Load ONNX model with onnxruntime and run quick verification."""
    if providers is None:
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    logger.info("Verifying %s with onnxruntime ...", onnx_path)
    session_options = ort.SessionOptions()
    session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL

    sess = ort.InferenceSession(str(onnx_path), sess_options=session_options, providers=providers)
    logger.info("ONNX Runtime using provider: %s", sess.get_providers()[0])

    model_output_names = [o.name for o in sess.get_outputs()]
    for name in expected_output_names:
        if name not in model_output_names:
            raise RuntimeError(
                f"Expected output '{name}' not found in model. "
                f"Available: {model_output_names}"
            )

    outputs = sess.run(expected_output_names, feeds)
    logger.info(
        "Verification passed — %d output(s) produced with shapes: %s",
        len(outputs),
        [o.shape for o in outputs],
    )
    return True


Overwriting /content/mobilei2v_code/model_utils.py


In [5]:
%%writefile /content/mobilei2v_code/models/mobiledit.py
#write to mobiledit.py
"""
Self-contained MobileI2V model architecture for forward pass and ONNX export.

Vendored from https://github.com/hustvl/MobileI2V
Apache-2.0 License

Architecture: Mobiledit_300M_P1_D16
  - PatchEmbed: 2D patch embedding
  - 16x SanaBlock (7 cross + 1 vanila + 7 cross + 1 vanila)
  - Attention: LiteLA (linear attention with RoPE3D)
  - FFN: GLUMBConv
  - Condition: t2i_modulate (adaLN-single)
  - Timestep embedder + Flow score embedder + Caption embedder
  - T2IFinalLayer: final modulation + linear projection

All dependencies (timm helpers, einops patterns) are vendored inline.
No xformers, no triton, no checkpointing, no distributed.
"""

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange


# ============================================================
# Utility functions
# ============================================================

def to_2tuple(x):
    if isinstance(x, (list, tuple)):
        return tuple(x)
    return (x, x)


def val2list(x, repeat_time=1):
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x for _ in range(repeat_time)]


def val2tuple(x, min_len=1, idx_repeat=-1):
    x = val2list(x)
    if len(x) > 0:
        x[idx_repeat:idx_repeat] = [x[idx_repeat] for _ in range(min_len - len(x))]
    return tuple(x)


def get_same_padding(kernel_size):
    if isinstance(kernel_size, tuple):
        return tuple(get_same_padding(ks) for ks in kernel_size)
    assert kernel_size % 2 > 0, f"kernel size {kernel_size} should be odd number"
    return kernel_size // 2


def auto_grad_checkpoint(module, *args, **kwargs):
    """Simplified: no checkpointing, just calls module directly."""
    return module(*args, **kwargs)


# ============================================================
# DropPath (from timm)
# ============================================================

class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) per sample (when applied in main path of residual blocks)."""

    def __init__(self, drop_prob=0.0, scale_by_keep=True):
        super().__init__()
        self.drop_prob = drop_prob
        self.scale_by_keep = scale_by_keep

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = x.new_empty(shape).bernoulli_(keep_prob)
        if keep_prob > 0.0 and self.scale_by_keep:
            random_tensor.div_(keep_prob)
        return x * random_tensor


# ============================================================
# RMSNorm
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, dim, scale_factor=1.0, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim) * scale_factor)

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        return (self.weight * self._norm(x.float())).type_as(x)


# ============================================================
# PositionGetter3D and RoPE3D
# ============================================================

class PositionGetter3D(object):
    """Return 3D positions of patches."""

    def __init__(self):
        self.cache_positions = {}

    def __call__(self, b, t, h, w, device):
        b_int, t_int, h_int, w_int = int(b), int(t), int(h), int(w)
        key = (b_int, t_int, h_int, w_int, str(device))
        if key not in self.cache_positions:
            x = torch.arange(w_int, device=device)
            y = torch.arange(h_int, device=device)
            z = torch.arange(t_int, device=device)
            Z, Y, X = torch.meshgrid(z, y, x, indexing='ij')
            pos = torch.stack([Z, Y, X], dim=0).reshape(3, -1).reshape(3, 1, -1).expand(3, b_int, -1).clone()
            poses = (pos[0].contiguous(), pos[1].contiguous(), pos[2].contiguous())
            max_poses = (t_int - 1, h_int - 1, w_int - 1)
            self.cache_positions[key] = (poses, max_poses)
        return self.cache_positions[key]


class RoPE3D(nn.Module):
    def __init__(self, freq=10000.0, F0=1.0, interpolation_scale_thw=(1, 1.4375, 2.5)):
        super().__init__()
        self.base = freq
        self.F0 = F0
        self.interpolation_scale_t = interpolation_scale_thw[0]
        self.interpolation_scale_h = interpolation_scale_thw[1]
        self.interpolation_scale_w = interpolation_scale_thw[2]
        self.cache = {}

    def get_cos_sin(self, D, seq_len, device, dtype, interpolation_scale=1):
        D_int, seq_len_int = int(D), int(seq_len)
        key = (D_int, seq_len_int, str(device), str(dtype), interpolation_scale)
        if key not in self.cache:
            inv_freq = 1.0 / (self.base ** (torch.arange(0, D_int, 2).float().to(device) / D_int))
            t = torch.arange(seq_len_int, device=device, dtype=inv_freq.dtype) / interpolation_scale
            freqs = torch.einsum("i,j->ij", t, inv_freq).to(dtype)
            freqs = torch.cat((freqs, freqs), dim=-1)
            cos = freqs.cos()
            sin = freqs.sin()
            self.cache[key] = (cos, sin)
        return self.cache[key]

    @staticmethod
    def rotate_half(x):
        x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)

    def apply_rope1d(self, tokens, pos1d, cos, sin):
        assert pos1d.ndim == 2
        cos = F.embedding(pos1d, cos)[:, None, :, :]
        sin = F.embedding(pos1d, sin)[:, None, :, :]
        return (tokens * cos) + (self.rotate_half(tokens) * sin)

    def forward(self, tokens, positions):
        assert tokens.size(3) % 3 == 0, "number of dimensions should be a multiple of three"
        D = tokens.size(3) // 3
        poses, max_poses = positions
        assert len(poses) == 3 and poses[0].ndim == 2
        cos_t, sin_t = self.get_cos_sin(
            D, max_poses[0] + 1, tokens.device, tokens.dtype, self.interpolation_scale_t
        )
        cos_y, sin_y = self.get_cos_sin(
            D, max_poses[1] + 1, tokens.device, tokens.dtype, self.interpolation_scale_h
        )
        cos_x, sin_x = self.get_cos_sin(
            D, max_poses[2] + 1, tokens.device, tokens.dtype, self.interpolation_scale_w
        )
        t, y, x = tokens.chunk(3, dim=-1)
        t = self.apply_rope1d(t, poses[0], cos_t, sin_t)
        y = self.apply_rope1d(y, poses[1], cos_y, sin_y)
        x = self.apply_rope1d(x, poses[2], cos_x, sin_x)
        tokens = torch.cat((t, y, x), dim=-1)
        return tokens


# ============================================================
# build_act (simplified)
# ============================================================

def build_act(name=None, **kwargs):
    if name is None or name.lower() == "none":
        return None
    name = name.lower()
    if name == "silu" or name == "swish":
        return nn.SiLU(**kwargs)
    elif name == "gelu":
        return nn.GELU(approximate="tanh")
    elif name == "relu":
        return nn.ReLU(**kwargs)
    elif name == "identity":
        return nn.Identity()
    else:
        raise ValueError(f"Unsupported activation: {name}")


# ============================================================
# LayerNorm2d + build_norm (simplified)
# ============================================================

class LayerNorm2d(nn.LayerNorm):
    def forward(self, x):
        out = x - torch.mean(x, dim=1, keepdim=True)
        out = out / torch.sqrt(torch.square(out).mean(dim=1, keepdim=True) + self.eps)
        if self.elementwise_affine:
            out = out * self.weight.view(1, -1, 1, 1) + self.bias.view(1, -1, 1, 1)
        return out


def build_norm(name=None, num_features=None, affine=True, **kwargs):
    if name is None or name.lower() == "none":
        return None
    name = name.lower()
    if name == "bn2d":
        return nn.BatchNorm2d(num_features, affine=affine)
    elif name == "ln":
        return nn.LayerNorm(num_features, elementwise_affine=affine)
    elif name == "ln2d":
        return LayerNorm2d(num_features, elementwise_affine=affine)
    else:
        raise ValueError(f"Unsupported norm: {name}")


# ============================================================
# ConvLayer
# ============================================================

class ConvLayer(nn.Module):
    def __init__(
        self,
        in_dim,
        out_dim,
        kernel_size=3,
        stride=1,
        dilation=1,
        groups=1,
        padding=None,
        use_bias=False,
        dropout=0.0,
        norm="bn2d",
        act="relu",
    ):
        super().__init__()
        if padding is None:
            padding = get_same_padding(kernel_size)
            padding *= dilation

        self.in_dim = in_dim
        self.out_dim = out_dim
        self.kernel_size = kernel_size
        self.stride = stride
        self.dilation = dilation
        self.groups = groups
        self.padding = padding
        self.use_bias = use_bias

        self.dropout = nn.Dropout2d(dropout, inplace=False) if dropout > 0 else None
        self.conv = nn.Conv2d(
            in_dim, out_dim,
            kernel_size=(kernel_size, kernel_size),
            stride=(stride, stride),
            padding=padding,
            dilation=(dilation, dilation),
            groups=groups,
            bias=use_bias,
        )
        self.norm = build_norm(norm, num_features=out_dim)
        self.act = build_act(act)

    def forward(self, x):
        if self.dropout is not None:
            x = self.dropout(x)
        x = self.conv(x)
        if self.norm is not None:
            x = self.norm(x)
        if self.act is not None:
            x = self.act(x)
        return x


# ============================================================
# GLUMBConv
# ============================================================

class GLUMBConv(nn.Module):
    def __init__(
        self,
        in_features,
        hidden_features,
        out_feature=None,
        kernel_size=3,
        stride=1,
        padding=None,
        use_bias=False,
        norm=(None, None, None),
        act=("silu", "silu", None),
        dilation=1,
    ):
        out_feature = out_feature or in_features
        super().__init__()
        use_bias = val2tuple(use_bias, 3)
        norm = val2tuple(norm, 3)
        act = val2tuple(act, 3)

        self.glu_act = build_act(act[1])

        self.inverted_conv = ConvLayer(
            in_features, hidden_features * 2, 1,
            use_bias=use_bias[0], norm=norm[0], act=act[0],
        )
        self.depth_conv = ConvLayer(
            hidden_features * 2, hidden_features * 2,
            kernel_size, stride=stride,
            groups=hidden_features * 2,
            padding=padding,
            use_bias=use_bias[1],
            norm=norm[1],
            act=None,
            dilation=dilation,
        )
        self.point_conv = ConvLayer(
            hidden_features, out_feature, 1,
            use_bias=use_bias[2], norm=norm[2], act=act[2],
        )

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.reshape(B, H, W, C).permute(0, 3, 1, 2)
        x = self.inverted_conv(x)
        x = self.depth_conv(x)
        x, gate = torch.chunk(x, 2, dim=1)
        gate = self.glu_act(gate)
        x = x * gate
        x = self.point_conv(x)
        x = x.reshape(B, C, N).permute(0, 2, 1)
        return x


# ============================================================
# Mlp (from timm, vendored inline)
# ============================================================

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, bias=True, drop=0.0):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features, bias=bias)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features, bias=bias)
        self.drop1 = nn.Dropout(drop)
        self.drop2 = nn.Dropout(drop)

    def forward(self, x, H=None, W=None):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x


class DWMlp(Mlp):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, bias=True, drop=0.0,
                 kernel_size=3, stride=1, dilation=1, padding=None):
        super().__init__(
            in_features=in_features,
            hidden_features=hidden_features,
            out_features=out_features,
            act_layer=act_layer,
            bias=bias,
            drop=drop,
        )
        hidden_features = hidden_features or in_features
        self.hidden_features = hidden_features
        if padding is None:
            padding = get_same_padding(kernel_size)
            padding *= dilation
        self.conv = nn.Conv2d(
            hidden_features, hidden_features,
            kernel_size=(kernel_size, kernel_size),
            stride=(stride, stride),
            padding=padding,
            dilation=(dilation, dilation),
            groups=hidden_features,
            bias=bias,
        )

    def forward(self, x, H=None, W=None):
        B, N, C = x.shape
        if H is None or W is None:
            H = W = int(N ** 0.5)
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = x.reshape(B, H, W, self.hidden_features).permute(0, 3, 1, 2)
        x = self.conv(x)
        x = x.reshape(B, self.hidden_features, N).permute(0, 2, 1)
        x = self.fc2(x)
        x = self.drop2(x)
        return x


# ============================================================
# t2i_modulate
# ============================================================

def t2i_modulate(x, shift, scale):
    return x * (1 + scale) + shift


def modulate(x, shift, scale):
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


# ============================================================
# MultiHeadCrossAttention (no xformers, vanilla fallback)
# ============================================================

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads, attn_drop=0.0, proj_drop=0.0, qk_norm=False, **block_kwargs):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.kv_linear = nn.Linear(d_model, d_model * 2)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(d_model, d_model)
        self.proj_drop = nn.Dropout(proj_drop)
        if qk_norm:
            self.q_norm = RMSNorm(d_model, scale_factor=1.0, eps=1e-6)
            self.k_norm = RMSNorm(d_model, scale_factor=1.0, eps=1e-6)
        else:
            self.q_norm = nn.Identity()
            self.k_norm = nn.Identity()

    def forward(self, x, cond, mask=None):
        B, N, C = x.shape
        q = self.q_linear(x)
        kv = self.kv_linear(cond).view(B, -1, 2, C)
        k, v = kv.unbind(2)
        q = self.q_norm(q).view(B, -1, self.num_heads, self.head_dim)
        k = self.k_norm(k).view(B, -1, self.num_heads, self.head_dim)
        v = v.view(B, -1, self.num_heads, self.head_dim)

        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        if mask is not None and mask.ndim == 2:
            mask = (1 - mask.to(q.dtype)) * -10000.0
            mask = mask[:, None, None].repeat(1, self.num_heads, 1, 1)
        x = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)
        x = x.transpose(1, 2)
        x = x.contiguous().view(B, -1, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


# ============================================================
# Vanilla Attention (no timm dependency, with RoPE3D)
# ============================================================

class Attention(nn.Module):
    """Vanilla multi-head self-attention with RoPE3D."""

    def __init__(self, dim, num_heads=8, qkv_bias=True, qk_norm=False, **block_kwargs):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(0.0)
        self.proj_drop = nn.Dropout(0.0)

        self.rope = RoPE3D(interpolation_scale_thw=(1, 1.4375, 2.5))
        self.position_getter = PositionGetter3D()

        if qk_norm:
            self.q_norm = RMSNorm(dim, scale_factor=1.0, eps=1e-5)
            self.k_norm = RMSNorm(dim, scale_factor=1.0, eps=1e-5)
        else:
            self.q_norm = nn.Identity()
            self.k_norm = nn.Identity()

    def _compute_rope_positions(self, q, T):
        import math
        B, H, N, D = q.shape
        S = N // T
        h_spatial = int(math.isqrt(S))
        w_spatial = (S + h_spatial - 1) // h_spatial
        pos_thw = self.position_getter(B, t=T, h=h_spatial, w=w_spatial, device=q.device)
        return pos_thw

    def forward(self, x, HW=None, T=None):
        B, N, C = x.shape

        qkv = self.qkv(x).reshape(B, N, 3, C)
        q, k, v = qkv.unbind(2)

        q = self.q_norm(q)
        k = self.k_norm(k)

        q = q.reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        k = k.reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        v = v.reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        # Apply RoPE3D
        if T is None:
            T = 3  # default fallback
        pos_thw = self._compute_rope_positions(q, T)
        q = self.rope(q, pos_thw)
        k = self.rope(k, pos_thw)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


# ============================================================
# LiteLA (Lightweight Linear Attention with RoPE3D)
# ============================================================

class LiteLA(nn.Module):
    """Lightweight linear attention with 3D RoPE."""

    PAD_VAL = 1

    def __init__(self, in_dim, out_dim, heads=None, heads_ratio=1.0,
                 dim=32, eps=1e-15, use_bias=False, qk_norm=False, norm_eps=1e-5):
        super().__init__()
        heads = heads or int(out_dim // dim * heads_ratio)
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.heads = heads
        self.dim = out_dim // heads
        self.eps = eps

        self.qkv = nn.Linear(in_dim, out_dim * 3, bias=use_bias)
        self.proj = nn.Linear(out_dim, out_dim)

        self.kernel_func = nn.ReLU(inplace=False)
        if qk_norm:
            self.q_norm = RMSNorm(in_dim, scale_factor=1.0, eps=norm_eps)
            self.k_norm = RMSNorm(in_dim, scale_factor=1.0, eps=norm_eps)
        else:
            self.q_norm = nn.Identity()
            self.k_norm = nn.Identity()

        self.rope = RoPE3D()
        self.position_getter = PositionGetter3D()

    def _compute_rope_positions(self, q, T):
        import math
        B, h, N, D = q.shape
        S = N // T
        h_spatial = int(math.isqrt(S))
        w_spatial = (S + h_spatial - 1) // h_spatial
        pos_thw = self.position_getter(B, t=T, h=h_spatial, w=w_spatial, device=q.device)
        return pos_thw

    def attn_matmul(self, q, k, v):
        q = self.kernel_func(q)
        k = self.kernel_func(k)

        v = F.pad(v, (0, 0, 0, 1), mode="constant", value=LiteLA.PAD_VAL)
        vk = torch.matmul(v, k)  # (B, h, h_d, N) @ (B, h, N, h_d) -> (B, h, h_d, h_d)
        out = torch.matmul(vk, q)  # (B, h, h_d, h_d) @ (B, h, h_d, N) -> (B, h, h_d, N)

        if out.dtype in [torch.float16, torch.bfloat16]:
            out = out.float()
        out = out[:, :, :-1] / (out[:, :, -1:] + self.eps)
        return out

    def forward(self, x, mask=None, HW=None, block_id=None, T=None):
        B, N, C = x.shape

        qkv = self.qkv(x).reshape(B, N, 3, C)
        q, k, v = qkv.unbind(2)
        dtype = q.dtype

        q = self.q_norm(q).transpose(-1, -2)  # (B, C, N)
        k = self.k_norm(k).transpose(-1, -2)  # (B, C, N)
        v = v.transpose(-1, -2)  # (B, C, N)

        q = q.reshape(B, C // self.dim, self.dim, N)          # (B, h, h_d, N)
        k = k.reshape(B, C // self.dim, self.dim, N).transpose(-1, -2)  # (B, h, N, h_d)
        v = v.reshape(B, C // self.dim, self.dim, N)          # (B, h, h_d, N)

        q = q.transpose(-1, -2)  # (B, h, N, h_d)

        # Apply RoPE3D
        if T is None:
            T = 3  # default fallback
        pos_thw = self._compute_rope_positions(q, T)
        q = self.rope(q, pos_thw)
        k = self.rope(k, pos_thw)

        q = q.transpose(-1, -2)  # (B, h, h_d, N)

        out = self.attn_matmul(q, k, v).to(dtype)

        out = out.view(B, C, N).permute(0, 2, 1)  # B, N, C
        out = self.proj(out)

        if torch.is_autocast_enabled() and torch.get_autocast_gpu_dtype() == torch.float16:
            out = out.clip(-65504, 65504)

        return out


# ============================================================
# FlashAttention (vanilla fallback, no xformers)
# ============================================================

class FlashAttention(nn.Module):
    """Multi-head attention using PyTorch's scaled_dot_product_attention."""

    def __init__(self, dim, num_heads=8, qkv_bias=True, qk_norm=False, **block_kwargs):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(0.0)
        self.proj_drop = nn.Dropout(0.0)

        if qk_norm:
            self.q_norm = nn.LayerNorm(dim)
            self.k_norm = nn.LayerNorm(dim)
        else:
            self.q_norm = nn.Identity()
            self.k_norm = nn.Identity()

    def forward(self, x, mask=None, HW=None, block_id=None, T=None):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, C)
        q, k, v = qkv.unbind(2)
        dtype = q.dtype

        q = self.q_norm(q)
        k = self.k_norm(k)

        q = q.reshape(B, N, self.num_heads, self.head_dim).to(dtype)
        k = k.reshape(B, N, self.num_heads, self.head_dim).to(dtype)
        v = v.reshape(B, N, self.num_heads, self.head_dim).to(dtype)

        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        if mask is not None and mask.ndim == 2:
            mask = (1 - mask.to(x.dtype)) * -10000.0
            mask = mask[:, None, None].repeat(1, self.num_heads, 1, 1)
        x = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)
        x = x.transpose(1, 2)

        x = x.contiguous().view(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)

        if torch.is_autocast_enabled() and torch.get_autocast_gpu_dtype() == torch.float16:
            x = x.clip(-65504, 65504)

        return x


# ============================================================
# PatchEmbed
# ============================================================

class PatchEmbed(nn.Module):
    """2D Image to Patch Embedding."""

    def __init__(self, img_height=224, img_width=224, patch_size=16,
                 in_chans=3, embed_dim=768, kernel_size=None,
                 padding=0, norm_layer=None, flatten=True, bias=True):
        super().__init__()
        kernel_size = kernel_size or patch_size
        patch_size = to_2tuple(patch_size)
        self.img_size = (img_height, img_width)
        self.patch_size = patch_size
        self.grid_size = (self.img_size[0] // patch_size[0], self.img_size[1] // patch_size[1])
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.flatten = flatten
        if not padding and kernel_size % 2 > 0:
            padding = get_same_padding(kernel_size)
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=kernel_size,
                              stride=patch_size, padding=padding, bias=bias)
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

    def forward(self, x):
        B, C, H, W = x.shape
        if self.flatten:
            x = self.proj(x).flatten(2).transpose(1, 2)
        else:
            x = self.proj(x)
        x = self.norm(x)
        return x


# ============================================================
# TimestepEmbedder
# ============================================================

class TimestepEmbedder(nn.Module):
    """Embeds scalar timesteps into vector representations."""

    def __init__(self, hidden_size, frequency_embedding_size=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size, bias=True),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size, bias=True),
        )
        self.frequency_embedding_size = frequency_embedding_size

    @staticmethod
    def timestep_embedding(t, dim, max_period=10000):
        half = dim // 2
        freqs = torch.exp(
            -math.log(max_period)
            * torch.arange(start=0, end=half, dtype=torch.float32, device=t.device)
            / half
        )
        args = t[:, :, None].float() * freqs[None]
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        if dim % 2:
            embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :, :1])], dim=-1)
        return embedding

    def forward(self, t):
        t_freq = self.timestep_embedding(t, self.frequency_embedding_size).to(self.dtype)
        t_emb = self.mlp(t_freq)
        return t_emb

    @property
    def dtype(self):
        try:
            return next(self.parameters()).dtype
        except StopIteration:
            return torch.float32


# ============================================================
# CaptionEmbedder
# ============================================================

class CaptionEmbedder(nn.Module):
    """Embeds text captions with classifier-free guidance dropout."""

    def __init__(self, in_channels, hidden_size, uncond_prob,
                 act_layer=nn.GELU(approximate="tanh"), token_num=120):
        super().__init__()
        self.y_proj = Mlp(
            in_features=in_channels, hidden_features=hidden_size,
            out_features=hidden_size, act_layer=act_layer, drop=0,
        )
        self.register_buffer(
            "y_embedding",
            nn.Parameter(torch.randn(token_num, in_channels) / in_channels ** 0.5),
        )
        self.uncond_prob = uncond_prob

    def token_drop(self, caption, force_drop_ids=None):
        if force_drop_ids is None:
            drop_ids = torch.rand(caption.shape[0]).to(caption.device) < self.uncond_prob
        else:
            drop_ids = force_drop_ids == 1
        caption = torch.where(drop_ids[:, None, None, None], self.y_embedding, caption)
        return caption

    def forward(self, caption, train, force_drop_ids=None):
        use_dropout = self.uncond_prob > 0
        if (train and use_dropout) or (force_drop_ids is not None):
            caption = self.token_drop(caption, force_drop_ids)
        caption = self.y_proj(caption)
        return caption


# ============================================================
# T2IFinalLayer
# ============================================================

class T2IFinalLayer(nn.Module):
    """Final layer of the diffusion transformer with modulation."""

    def __init__(self, hidden_size, patch_size, out_channels):
        super().__init__()
        self.norm_final = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.linear = nn.Linear(hidden_size, patch_size * patch_size * out_channels, bias=True)
        self.scale_shift_table = nn.Parameter(torch.randn(2, hidden_size) / hidden_size ** 0.5)
        self.out_channels = out_channels

    def forward(self, x, t):
        shift, scale = (self.scale_shift_table[None] + t[:, :, None]).chunk(2, dim=2)
        shift = shift.squeeze(2)
        scale = scale.squeeze(2)
        x = t2i_modulate(self.norm_final(x), shift, scale)
        x = self.linear(x)
        return x


# ============================================================
# SanaBlock_cross
# ============================================================

class SanaBlock_cross(nn.Module):
    """
    Transformer block with self-attention + MLP.
    Uses adaLN-single (t2i_modulate) conditioning.
    """

    def __init__(
        self,
        hidden_size,
        num_heads,
        mlp_ratio=4.0,
        drop_path=0,
        input_size=None,
        qk_norm=False,
        attn_type="flash",
        ffn_type="mlp",
        mlp_acts=("silu", "silu", None),
        linear_head_dim=32,
        **block_kwargs,
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)

        if attn_type == "flash":
            self.attn = FlashAttention(hidden_size, num_heads=num_heads, qkv_bias=True, qk_norm=qk_norm, **block_kwargs)
        elif attn_type == "linear":
            linear_head_dim = 72
            self_num_heads = hidden_size // linear_head_dim
            self.attn = LiteLA(hidden_size, hidden_size, heads=self_num_heads, eps=1e-8, qk_norm=qk_norm)
        elif attn_type == "vanilla":
            self.attn = Attention(hidden_size, num_heads=num_heads, qkv_bias=True, qk_norm=qk_norm)
        else:
            raise ValueError(f"Unsupported attn_type: {attn_type}")

        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)

        if ffn_type == "dwmlp":
            approx_gelu = lambda: nn.GELU(approximate="tanh")
            self.mlp = DWMlp(
                in_features=hidden_size, hidden_features=int(hidden_size * mlp_ratio),
                act_layer=approx_gelu, drop=0,
            )
        elif ffn_type == "glumbconv":
            self.mlp = GLUMBConv(
                in_features=hidden_size,
                hidden_features=int(hidden_size * mlp_ratio),
                use_bias=(True, True, False),
                norm=(None, None, None),
                act=mlp_acts,
            )
        elif ffn_type == "glumbconv_dilate":
            self.mlp = GLUMBConv(
                in_features=hidden_size,
                hidden_features=int(hidden_size * mlp_ratio),
                use_bias=(True, True, False),
                norm=(None, None, None),
                act=mlp_acts,
                dilation=2,
            )
        elif ffn_type == "mlp":
            approx_gelu = lambda: nn.GELU(approximate="tanh")
            self.mlp = Mlp(
                in_features=hidden_size, hidden_features=int(hidden_size * mlp_ratio),
                act_layer=approx_gelu, drop=0,
            )
        else:
            raise ValueError(f"Unsupported ffn_type: {ffn_type}")

        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.scale_shift_table = nn.Parameter(torch.randn(6, hidden_size) / hidden_size ** 0.5)

    def forward(self, x, y, t, flow_score, mask=None, H=None, W=None, T=None, S=None, **kwargs):
        B, N, C = x.shape

        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
            self.scale_shift_table[None] + t.reshape(B, N, 6, -1) + flow_score.reshape(B, N, 6, -1)
        ).chunk(6, dim=2)
        shift_msa = shift_msa.squeeze(2)
        scale_msa = scale_msa.squeeze(2)
        gate_msa = gate_msa.squeeze(2)
        shift_mlp = shift_mlp.squeeze(2)
        scale_mlp = scale_mlp.squeeze(2)
        gate_mlp = gate_mlp.squeeze(2)

        # Self-attention with adaLN modulation
        x_m = t2i_modulate(self.norm1(x), shift_msa, scale_msa)
        x_s = self.attn(x_m, HW=(H, W) if H is not None and W is not None else None, T=T)
        x_s = gate_msa * x_s
        x = x + self.drop_path(x_s)

        # MLP with adaLN modulation and temporal rearrangement
        x_m = t2i_modulate(self.norm2(x), shift_mlp, scale_mlp)
        x_m = rearrange(x_m, "B (T S) C -> (B T) S C", T=T, S=S)
        x_mlp = self.mlp(x_m, H, W)
        x_mlp = rearrange(x_mlp, "(B T) S C -> B (T S) C", T=T, S=S)
        x_mlp = gate_mlp * x_mlp
        x = x + self.drop_path(x_mlp)

        return x


# ============================================================
# SanaBlock_vanila
# ============================================================

class SanaBlock_vanila(nn.Module):
    """
    Transformer block with vanilla self-attention + MLP.
    Uses adaLN-single (t2i_modulate) conditioning.
    """

    def __init__(
        self,
        hidden_size,
        num_heads,
        mlp_ratio=4.0,
        drop_path=0,
        input_size=None,
        qk_norm=False,
        attn_type="flash",
        ffn_type="mlp",
        mlp_acts=("silu", "silu", None),
        linear_head_dim=32,
        **block_kwargs,
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn = Attention(hidden_size, num_heads=num_heads, qkv_bias=True, qk_norm=qk_norm)

        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)

        if ffn_type == "dwmlp":
            approx_gelu = lambda: nn.GELU(approximate="tanh")
            self.mlp = DWMlp(
                in_features=hidden_size, hidden_features=int(hidden_size * mlp_ratio),
                act_layer=approx_gelu, drop=0,
            )
        elif ffn_type == "glumbconv":
            self.mlp = GLUMBConv(
                in_features=hidden_size,
                hidden_features=int(hidden_size * mlp_ratio),
                use_bias=(True, True, False),
                norm=(None, None, None),
                act=mlp_acts,
            )
        elif ffn_type == "glumbconv_dilate":
            self.mlp = GLUMBConv(
                in_features=hidden_size,
                hidden_features=int(hidden_size * mlp_ratio),
                use_bias=(True, True, False),
                norm=(None, None, None),
                act=mlp_acts,
                dilation=2,
            )
        elif ffn_type == "mlp":
            approx_gelu = lambda: nn.GELU(approximate="tanh")
            self.mlp = Mlp(
                in_features=hidden_size, hidden_features=int(hidden_size * mlp_ratio),
                act_layer=approx_gelu, drop=0,
            )
        else:
            raise ValueError(f"Unsupported ffn_type: {ffn_type}")

        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.scale_shift_table = nn.Parameter(torch.randn(6, hidden_size) / hidden_size ** 0.5)

    def forward(self, x, y, t, flow_score, mask=None, H=None, W=None, T=None, S=None, **kwargs):
        B, N, C = x.shape

        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
            self.scale_shift_table[None] + t.reshape(B, N, 6, -1) + flow_score.reshape(B, N, 6, -1)
        ).chunk(6, dim=2)
        shift_msa = shift_msa.squeeze(2)
        scale_msa = scale_msa.squeeze(2)
        gate_msa = gate_msa.squeeze(2)
        shift_mlp = shift_mlp.squeeze(2)
        scale_mlp = scale_mlp.squeeze(2)
        gate_mlp = gate_mlp.squeeze(2)

        # Self-attention with adaLN modulation
        x_m = t2i_modulate(self.norm1(x), shift_msa, scale_msa)
        x_s = self.attn(x_m, HW=(H, W) if H is not None and W is not None else None, T=T)
        x_s = gate_msa * x_s
        x = x + self.drop_path(x_s)

        # MLP with adaLN modulation and temporal rearrangement
        x_m = t2i_modulate(self.norm2(x), shift_mlp, scale_mlp)
        x_m = rearrange(x_m, "B (T S) C -> (B T) S C", T=T, S=S)
        x_mlp = self.mlp(x_m, H, W)
        x_mlp = rearrange(x_mlp, "(B T) S C -> B (T S) C", T=T, S=S)
        x_mlp = gate_mlp * x_mlp
        x = x + self.drop_path(x_mlp)

        return x


# ============================================================
# Position embedding helpers
# ============================================================

def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):
    assert embed_dim % 2 == 0
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])
    emb = np.concatenate([emb_h, emb_w], axis=1)
    return emb


def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=np.float64)
    omega /= embed_dim / 2.0
    omega = 1.0 / 10000 ** omega

    pos = pos.reshape(-1)
    out = np.einsum("m,d->md", pos, omega)

    emb_sin = np.sin(out)
    emb_cos = np.cos(out)
    emb = np.concatenate([emb_sin, emb_cos], axis=1)
    return emb


def get_2d_sincos_pos_embed(embed_dim, grid_size, cls_token=False, extra_tokens=0,
                            pe_interpolation=1.0, base_size=(16, 16)):
    if isinstance(grid_size, int):
        grid_size = to_2tuple(grid_size)
    grid_h = np.arange(grid_size[0], dtype=np.float32) / (grid_size[0] / base_size[0]) / pe_interpolation
    grid_w = np.arange(grid_size[1], dtype=np.float32) / (grid_size[1] / base_size[1]) / pe_interpolation
    grid = np.meshgrid(grid_w, grid_h)
    grid = np.stack(grid, axis=0)
    grid = grid.reshape([2, 1, grid_size[1], grid_size[0]])

    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)
    if cls_token and extra_tokens > 0:
        pos_embed = np.concatenate([np.zeros([extra_tokens, embed_dim]), pos_embed], axis=0)
    return pos_embed


def get_1d_sincos_pos_embed(embed_dim, length, scale=1.0):
    pos = np.arange(0, length)[..., None] / scale
    return get_1d_sincos_pos_embed_from_grid(embed_dim, pos)


# ============================================================
# Main Mobiledit Model
# ============================================================

class Mobiledit(nn.Module):
    """
    MobileI2V diffusion model with transformer backbone.

    Input:  (B, C, T, H, W) latent video
    Output: (B, C_out, T, H, W) predicted noise/denoised latents
    """

    def __init__(
        self,
        input_height=32,
        input_width=32,
        patch_size=2,
        in_channels=4,
        hidden_size=1152,
        depth=28,
        num_heads=16,
        mlp_ratio=4.0,
        class_dropout_prob=0.1,
        pred_sigma=True,
        drop_path=0.0,
        caption_channels=2304,
        pe_interpolation=1.0,
        config=None,
        model_max_length=120,
        qk_norm=False,
        y_norm=False,
        norm_eps=1e-5,
        attn_type="flash",
        ffn_type="mlp",
        use_pe=True,
        y_norm_scale_factor=1.0,
        patch_embed_kernel=None,
        mlp_acts=("silu", "silu", None),
        linear_head_dim=32,
        **kwargs,
    ):
        super().__init__()
        self.pred_sigma = pred_sigma
        self.in_channels = in_channels
        self.out_channels = in_channels * 2 if pred_sigma else in_channels
        self.patch_size = patch_size
        self.num_heads = num_heads
        self.pe_interpolation = pe_interpolation
        self.depth = depth
        self.use_pe = use_pe
        self.y_norm = y_norm
        self.input_size = (17, input_height, input_width)
        self.hidden_size = hidden_size

        num_patches = np.prod([self.input_size[i] // 1 for i in range(3)])
        self.num_patches = num_patches
        self.num_temporal = self.input_size[0] // 1
        self.num_spatial = num_patches // self.num_temporal

        kernel_size = patch_embed_kernel or patch_size

        self.x_embedder = PatchEmbed(
            input_height, input_width, patch_size, in_channels, hidden_size,
            kernel_size=kernel_size, bias=True,
        )

        self.t_embedder = TimestepEmbedder(hidden_size)
        self.flow_embedder = TimestepEmbedder(hidden_size)
        num_patches = self.x_embedder.num_patches
        self.base_size = (input_height // self.patch_size, input_width // self.patch_size)

        self.register_buffer("pos_embed", self.get_spatial_pos_embed())
        self.register_buffer("pos_embed_temporal", self.get_temporal_pos_embed())

        approx_gelu = lambda: nn.GELU(approximate="tanh")
        self.t_block = nn.Sequential(nn.SiLU(), nn.Linear(hidden_size, 6 * hidden_size, bias=True))
        self.flow_block = nn.Sequential(nn.SiLU(), nn.Linear(hidden_size, 6 * hidden_size, bias=True))
        self.y_embedder = CaptionEmbedder(
            in_channels=caption_channels,
            hidden_size=hidden_size,
            uncond_prob=class_dropout_prob,
            act_layer=approx_gelu,
            token_num=model_max_length,
        )
        if self.y_norm:
            self.attention_y_norm = RMSNorm(hidden_size, scale_factor=y_norm_scale_factor, eps=norm_eps)

        drop_path = [x.item() for x in torch.linspace(0, drop_path, depth + 2)]

        # Block arrangement: 7 cross + 1 vanila + 7 cross + 1 vanila
        self.blocks = nn.ModuleList(
            [
                SanaBlock_cross(
                    hidden_size, num_heads, mlp_ratio=mlp_ratio,
                    drop_path=drop_path[i],
                    input_size=(input_height // patch_size, input_width // patch_size),
                    qk_norm=qk_norm, attn_type=attn_type, ffn_type=ffn_type,
                    mlp_acts=mlp_acts, linear_head_dim=linear_head_dim,
                )
                for i in range(7)
            ]
        )
        for _ in range(1):
            self.blocks.append(
                SanaBlock_vanila(
                    hidden_size, num_heads, mlp_ratio=mlp_ratio,
                    drop_path=drop_path[14],
                    input_size=(input_height // patch_size, input_width // patch_size),
                    attn_type=attn_type, ffn_type=ffn_type,
                    mlp_acts=mlp_acts, linear_head_dim=linear_head_dim,
                )
            )
        for i in range(7):
            self.blocks.append(
                SanaBlock_cross(
                    hidden_size, num_heads, mlp_ratio=mlp_ratio,
                    drop_path=drop_path[i],
                    input_size=(input_height // patch_size, input_width // patch_size),
                    qk_norm=qk_norm, attn_type=attn_type, ffn_type=ffn_type,
                    mlp_acts=mlp_acts, linear_head_dim=linear_head_dim,
                )
            )
        for _ in range(1):
            self.blocks.append(
                SanaBlock_vanila(
                    hidden_size, num_heads, mlp_ratio=mlp_ratio,
                    drop_path=drop_path[14],
                    input_size=(input_height // patch_size, input_width // patch_size),
                    attn_type=attn_type, ffn_type=ffn_type,
                    mlp_acts=mlp_acts, linear_head_dim=linear_head_dim,
                )
            )

        self.final_layer = T2IFinalLayer(hidden_size, patch_size, self.out_channels)
        self.initialize_weights()

    def get_dynamic_size(self, x):
        _, _, T, H, W = x.size()
        if T % self.patch_size != 0:
            T += self.patch_size - T % self.patch_size
        if H % self.patch_size != 0:
            H += self.patch_size - H % self.patch_size
        if W % self.patch_size != 0:
            W += self.patch_size - W % self.patch_size
        T = T // self.patch_size
        H = H // self.patch_size
        W = W // self.patch_size
        return T, H, W

    def forward(self, x, timestep, guide_image, y, cond_mask, flow_score,
                mask=None, data_info=None, **kwargs):
        """
        Forward pass of Mobiledit.

        Args:
            x: (B, C, T, H, W) latent video
            timestep: (B, N) or (B,) timestep values
            guide_image: (B, C, 1, H, W) first frame (guide) image
            y: (B, 1, L, C) text embeddings
            cond_mask: (B,) conditioning mask
            flow_score: (B,) flow score values
            mask: optional attention mask
            data_info: optional dict with extra info
        """
        B = x.shape[0]
        x = x.to(self.dtype)
        timestep = timestep.to(self.dtype)
        y = y.to(self.dtype)

        _, _, Tx, Hx, Wx = x.size()
        T, H, W = self.get_dynamic_size(x)
        S = H * W

        # Patch embed: (B, C, T, H, W) -> (B*T, C, H, W) -> (B*T, S, D) -> (B, T*S, D)
        x = rearrange(x, "B C T H W -> (B T) C H W")
        x = self.x_embedder(x)  # (B*T, S, D)
        x = rearrange(x, "(B T) S C -> B (T S) C", B=B, T=T, S=S)

        # Timestep and flow score conditioning
        num_patches = T * S
        timestep = timestep.unsqueeze(1).repeat(1, num_patches)
        cond_mask = cond_mask.unsqueeze(1).repeat(1, num_patches)
        timestep = torch.min(timestep, (1.0 - cond_mask) * 1000)
        timestep = timestep / 1000.0
        t = self.t_embedder(timestep.to(x.dtype))  # (B, N, D)
        t0 = self.t_block(t)  # (B, N, 6*D)

        flow_score = flow_score.unsqueeze(1).repeat(1, num_patches)
        flow_score_emb = self.flow_embedder(flow_score.to(x.dtype))  # (B, N, D)
        flow_score_emb = self.flow_block(flow_score_emb)  # (B, N, 6*D)

        # Text embedding
        y = self.y_embedder(y, self.training)  # (B, 1, L, D)
        if self.y_norm:
            y = self.attention_y_norm(y)

        if mask is not None:
            if mask.shape[0] != y.shape[0]:
                mask = mask.repeat(y.shape[0] // mask.shape[0], 1)
            mask = mask.squeeze(1).squeeze(1)
            y = y.squeeze(1).masked_select(mask.unsqueeze(-1) != 0).view(1, -1, x.shape[-1])
            y_lens = mask.sum(dim=1).tolist()
        else:
            y_lens = [y.shape[2]] * y.shape[0]
            y = y.squeeze(1).view(1, -1, x.shape[-1])

        # Transformer blocks
        for block in self.blocks:
            x = auto_grad_checkpoint(
                block, x, y, t0, flow_score_emb, y_lens, H, W, T, S,
            )

        # Final layer
        x = self.final_layer(x, t)  # (B, N, patch_size^2 * out_channels)
        x = self.unpatchify(x, T, H, W, Tx, Hx, Wx)

        return x

    def unpatchify(self, x, N_t, N_h, N_w, R_t, R_h, R_w):
        """
        Convert patch tokens back to latent video.

        Args:
            x: (B, N_t*N_h*N_w, patch_size^2 * C_out)
            N_t, N_h, N_w: number of patches per dimension
            R_t, R_h, R_w: actual temporal, height, width (before padding)
        """
        T_p = H_p = W_p = self.patch_size
        x = rearrange(
            x,
            "B (N_t N_h N_w) (T_p H_p W_p C_out) -> B C_out (N_t T_p) (N_h H_p) (N_w W_p)",
            N_t=N_t, N_h=N_h, N_w=N_w,
            T_p=T_p, H_p=H_p, W_p=W_p,
            C_out=self.out_channels,
        )
        # Unpad to original size
        x = x[:, :, :R_t, :R_h, :R_w]
        return x

    def get_spatial_pos_embed(self, grid_size=None):
        if grid_size is None:
            grid_size = self.input_size[1:]
        pos_embed = get_2d_sincos_pos_embed(
            self.hidden_size,
            (grid_size[0] // self.patch_size, grid_size[1] // self.patch_size),
            pe_interpolation=1,
            base_size=self.base_size,
        )
        pos_embed = torch.from_numpy(pos_embed).float().unsqueeze(0).requires_grad_(False)
        return pos_embed

    def get_temporal_pos_embed(self):
        pos_embed = get_1d_sincos_pos_embed(
            self.hidden_size,
            self.input_size[0] // self.patch_size,
            scale=1.0,
        )
        pos_embed = torch.from_numpy(pos_embed).float().unsqueeze(0).requires_grad_(False)
        return pos_embed

    def initialize_weights(self):
        def _basic_init(module):
            if isinstance(module, nn.Linear):
                torch.nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

        self.apply(_basic_init)

        if self.use_pe:
            pos_embed = get_2d_sincos_pos_embed(
                self.pos_embed.shape[-1],
                int(self.x_embedder.num_patches ** 0.5),
                pe_interpolation=self.pe_interpolation,
                base_size=self.base_size,
            )
            self.pos_embed.data.copy_(torch.from_numpy(pos_embed).float().unsqueeze(0))

        # Initialize patch_embed like nn.Linear (instead of nn.Conv2d):
        w = self.x_embedder.proj.weight.data
        nn.init.xavier_uniform_(w.view([w.shape[0], -1]))

        # Initialize timestep embedding MLP:
        nn.init.normal_(self.t_embedder.mlp[0].weight, std=0.02)
        nn.init.normal_(self.t_embedder.mlp[2].weight, std=0.02)
        nn.init.normal_(self.t_block[1].weight, std=0.02)

        # Initialize flow embedding MLP:
        nn.init.normal_(self.flow_embedder.mlp[0].weight, std=0.02)
        nn.init.normal_(self.flow_embedder.mlp[2].weight, std=0.02)
        nn.init.normal_(self.flow_block[1].weight, std=0.02)

        # Initialize caption embedding MLP:
        nn.init.normal_(self.y_embedder.y_proj.fc1.weight, std=0.02)
        nn.init.normal_(self.y_embedder.y_proj.fc2.weight, std=0.02)

    @property
    def dtype(self):
        return next(self.parameters()).dtype


# ============================================================
# Factory function
# ============================================================

def mobiledit_300m_P1_D16(**kwargs):
    """Create Mobiledit-300M with patch_size=1 and depth=16."""
    return Mobiledit(
        depth=16, hidden_size=1152, patch_size=1, num_heads=16,
        in_channels=128,
        pred_sigma=False,
        caption_channels=896,
        model_max_length=300,
        **kwargs,
    )


# ============================================================
# ONNX-friendly wrapper
# ============================================================

class MobileditONNXWrapper(nn.Module):
    """ONNX-friendly wrapper around Mobiledit.

    Exposes a clean forward(latent, text_emb, timestep) interface.
    guide_image and flow_score are injected as constants.
    """

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, latent, text_emb, timestep):
        """
        Args:
            latent: (B, C, T, H, W) latent video tensor
            text_emb: (B, 1, L, C) text embeddings
            timestep: (B,) or (B, 1) diffusion timestep
        Returns:
            (B, C_out, T, H, W) predicted denoised latent
        """
        B = latent.shape[0]

        # guide_image: use first frame of latent
        guide_image = latent[:, :, :1, :, :]

        # cond_mask: always 1.0 (fully conditioned)
        cond_mask = torch.ones(B, device=latent.device, dtype=latent.dtype)

        # flow_score: default 2.0
        flow_score = torch.full((B,), 2.0, device=latent.device, dtype=latent.dtype)

        return self.model(latent, timestep, guide_image, text_emb, cond_mask, flow_score)



In [6]:
%%writefile /content/mobilei2v_code/models/__init__.py
from .mobiledit import Mobiledit, mobiledit_300m_P1_D16, MobileditONNXWrapper
from .turbo_vaed_model import build_turbo_vaed_decoder, TurboVAEDDecoder3d


In [ ]:
%%writefile /content/mobilei2v_code/models/turbo_vaed_model.py
"""
turbo_vaed_model.py — Vendored Turbo-VAED Decoder model for ONNX export.

Self-contained implementation of all Turbo-VAED custom blocks, extracted from:
  https://github.com/hustvl/Turbo-VAED/blob/main/diffusers_vae/src/diffusers/models/autoencoders/autoencoder_kl_turbo_vaed.py

Apache-2.0 License. Original work by HUST Vision Lab / HuggingFace.

Usage:
    from models.turbo_vaed_model import build_turbo_vaed_decoder

    decoder = build_turbo_vaed_decoder(config_dict)
    state_dict = torch.load("Turbo-VAED-LTX.pth", map_location="cpu")
    decoder.load_state_dict(state_dict, strict=False)
    decoder.eval()
"""

from __future__ import annotations

from typing import Optional, Tuple, Union

import torch
import torch.nn as nn


# ============================================================
# Vendored utility classes (from diffusers_vae)
# ============================================================

class RMSNorm(nn.Module):
    """RMS Normalization as used by Turbo-VAED."""

    def __init__(self, dim: int, eps: float = 1e-6, elementwise_affine: bool = True):
        super().__init__()
        self.eps = eps
        if elementwise_affine:
            self.weight = nn.Parameter(torch.ones(dim))
        else:
            self.weight = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype
        variance = x.to(torch.float32).pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        if self.weight is not None:
            x = x * self.weight.to(x.dtype)
        return x.to(input_dtype)


def get_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "swish" or name == "silu":
        return nn.SiLU()
    elif name == "relu":
        return nn.ReLU()
    elif name == "gelu":
        return nn.GELU()
    else:
        raise ValueError(f"Unknown activation: {name}")


# ============================================================
# Custom Conv Blocks
# ============================================================

class TurboVAEDConv2dSplitUpsampler(nn.Module):
    """2D pixel-shuffle upsampler (spatial only, no temporal dim)."""

    def __init__(
        self,
        in_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding_mode: str = "zeros",
    ):
        super().__init__()
        self.stride = (stride, stride) if isinstance(stride, int) else stride
        self.kernel_size = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size

        height_pad = self.kernel_size[0] // 2
        width_pad = self.kernel_size[1] // 2
        padding = (height_pad, width_pad)

        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=self.kernel_size,
            stride=1,
            padding=padding,
            padding_mode=padding_mode,
        )

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = self.conv(hidden_states)
        hidden_states = torch.nn.functional.pixel_shuffle(hidden_states, self.stride[0])
        return hidden_states


class TurboVAEDConv2dUpsampler(nn.Module):
    """2D pixel-shuffle upsampler (applied frame-by-frame along temporal dim)."""

    def __init__(
        self,
        in_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding_mode: str = "zeros",
    ):
        super().__init__()
        self.stride = (stride, stride) if isinstance(stride, int) else stride
        self.kernel_size = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.in_channels = in_channels

        height_pad = self.kernel_size[0] // 2
        width_pad = self.kernel_size[1] // 2
        padding = (height_pad, width_pad)

        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=self.kernel_size,
            stride=1,
            padding=padding,
            padding_mode=padding_mode,
        )

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # [B, C, T, H, W]
        batch_size, channels, time_steps, height, width = hidden_states.shape
        # Flatten batch+time for 2D conv
        hidden_states = hidden_states.permute(0, 2, 1, 3, 4).reshape(batch_size * time_steps, channels, height, width)
        hidden_states = self.conv(hidden_states)
        hidden_states = torch.nn.functional.pixel_shuffle(hidden_states, self.stride[0])
        _, _, output_height, output_width = hidden_states.shape
        output_channels = hidden_states.shape[1]
        hidden_states = hidden_states.reshape(batch_size, time_steps, output_channels, output_height, output_width)
        hidden_states = hidden_states.permute(0, 2, 1, 3, 4)
        return hidden_states


class TurboVAEDCausalConv3d(nn.Module):
    """3D causal or non-causal convolution used in Turbo-VAED."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        dilation: int = 1,
        groups: int = 1,
        padding_mode: str = "zeros",
        is_causal: bool = True,
    ):
        super().__init__()
        self.is_causal = is_causal
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size, kernel_size)
        dilation = dilation if isinstance(dilation, tuple) else (dilation, 1, 1)
        stride = stride if isinstance(stride, tuple) else (stride, stride, stride)

        height_pad = self.kernel_size[1] // 2
        width_pad = self.kernel_size[2] // 2
        padding = (0, height_pad, width_pad)  # temporal padding done manually

        self.conv = nn.Conv3d(
            in_channels, out_channels,
            self.kernel_size,
            stride=stride,
            dilation=dilation,
            groups=groups,
            padding=padding,
            padding_mode=padding_mode,
        )

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        time_kernel_size = self.kernel_size[0]
        if self.is_causal:
            if time_kernel_size > 1:
                pad_left = hidden_states[:, :, :1, :, :].repeat((1, 1, time_kernel_size - 1, 1, 1))
                hidden_states = torch.cat([pad_left, hidden_states], dim=2)
        else:
            if time_kernel_size > 1:
                pad_count = (time_kernel_size - 1) // 2
                pad_left = hidden_states[:, :, :1, :, :].repeat((1, 1, pad_count, 1, 1))
                pad_right = hidden_states[:, :, -1:, :, :].repeat((1, 1, pad_count, 1, 1))
                hidden_states = torch.cat([pad_left, hidden_states, pad_right], dim=2)
        hidden_states = self.conv(hidden_states)
        return hidden_states


class TurboVAEDCausalDepthwiseSeperableConv3d(nn.Module):
    """Depthwise-separable 3D convolution."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        dilation: int = 1,
        padding_mode: str = "zeros",
        is_causal: bool = True,
    ):
        super().__init__()
        self.is_causal = is_causal
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size, kernel_size)
        self.stride = stride if isinstance(stride, tuple) else (stride, stride, stride)
        self.dilation = dilation if isinstance(dilation, tuple) else (dilation, 1, 1)

        height_pad = self.kernel_size[1] // 2
        width_pad = self.kernel_size[2] // 2
        padding = (0, height_pad, width_pad)

        self.depthwise_conv = nn.Conv3d(
            in_channels, in_channels,
            self.kernel_size,
            stride=self.stride,
            dilation=self.dilation,
            groups=in_channels,
            padding=padding,
            padding_mode=padding_mode,
        )
        self.pointwise_conv = nn.Conv3d(
            in_channels, out_channels,
            kernel_size=1,
        )

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        time_kernel_size = self.kernel_size[0]
        if time_kernel_size > 1:
            pad_count = (time_kernel_size - 1) // 2
            pad_left = hidden_states[:, :, :1, :, :].repeat((1, 1, pad_count, 1, 1))
            pad_right = hidden_states[:, :, -1:, :, :].repeat((1, 1, pad_count, 1, 1))
            hidden_states = torch.cat([pad_left, hidden_states, pad_right], dim=2)

        hidden_states = self.depthwise_conv(hidden_states)
        hidden_states = self.pointwise_conv(hidden_states)
        return hidden_states


# ============================================================
# TurboVAEDUpsampler3d — decoupled 3D pixel shuffle
# ============================================================

class TurboVAEDUpsampler3d(nn.Module):
    """Decoupled 3D pixel shuffle upsampler: temporal + spatial."""

    def __init__(
        self,
        in_channels: int,
        stride: int = 1,
        is_causal: bool = True,
        residual: bool = False,
        upscale_factor: int = 1,
        padding_mode: str = "zeros",
        is_video_dc_ae: bool = False,
    ):
        super().__init__()
        self.stride = stride if isinstance(stride, tuple) else (stride, stride, stride)
        self.residual = residual
        self.upscale_factor = upscale_factor
        self.is_video_dc_ae = is_video_dc_ae

        out_channels = (in_channels * stride[0] * stride[1] * stride[2]) // upscale_factor

        self.conv = TurboVAEDCausalConv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=1,
            is_causal=is_causal,
            padding_mode=padding_mode,
        )

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        batch_size, num_channels, num_frames, height, width = hidden_states.shape

        hidden_states = self.conv(hidden_states)

        # Step 1: temporal upsampling via reshape+permute
        hidden_states = hidden_states.reshape(batch_size, -1, self.stride[0], num_frames, height, width)
        hidden_states = hidden_states.permute(0, 1, 3, 2, 4, 5)
        hidden_states = hidden_states.reshape(batch_size, -1, num_frames * self.stride[0], height, width)

        # Step 2: spatial 2D pixel shuffle
        upsampled_frames = num_frames * self.stride[0]
        hidden_states = hidden_states.permute(0, 2, 1, 3, 4)
        hidden_states = hidden_states.reshape(batch_size * upsampled_frames, -1, height, width)
        hidden_states = torch.nn.functional.pixel_shuffle(hidden_states, self.stride[1])

        # Step 3: reshape back
        _, c, h, w = hidden_states.shape
        hidden_states = hidden_states.reshape(batch_size, upsampled_frames, c, h, w)
        hidden_states = hidden_states.permute(0, 2, 1, 3, 4)

        # Step 4: remove temporal padding
        if not self.is_video_dc_ae:
            hidden_states = hidden_states[:, :, self.stride[0] - 1:]

        return hidden_states


# ============================================================
# TurboVAEDResnetBlock3d
# ============================================================

class TurboVAEDResnetBlock3d(nn.Module):
    """3D ResNet block with optional depthwise separable conv and timestep conditioning."""

    def __init__(
        self,
        in_channels: int,
        out_channels: Optional[int] = None,
        dropout: float = 0.0,
        eps: float = 1e-6,
        elementwise_affine: bool = False,
        non_linearity: str = "swish",
        is_causal: bool = True,
        inject_noise: bool = False,
        timestep_conditioning: bool = False,
        is_upsampler_modified: bool = False,
        is_dw_conv: bool = False,
        dw_kernel_size: int = 3,
    ):
        super().__init__()

        out_channels = out_channels or in_channels
        self.nonlinearity = get_activation(non_linearity)
        conv_op = TurboVAEDCausalDepthwiseSeperableConv3d if is_dw_conv else TurboVAEDCausalConv3d
        kernel_size = dw_kernel_size if is_dw_conv else 3

        self.is_upsampler_modified = is_upsampler_modified
        self.replace_nonlinearity = get_activation("relu")

        self.norm1 = RMSNorm(in_channels, eps=1e-8, elementwise_affine=elementwise_affine)
        self.conv1 = conv_op(
            in_channels=in_channels, out_channels=out_channels,
            kernel_size=kernel_size, is_causal=is_causal,
        )

        self.norm2 = RMSNorm(out_channels, eps=1e-8, elementwise_affine=elementwise_affine)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = conv_op(
            in_channels=out_channels, out_channels=out_channels,
            kernel_size=kernel_size, is_causal=is_causal,
        )

        self.norm3 = None
        self.conv_shortcut = None
        if in_channels != out_channels:
            self.norm3 = nn.LayerNorm(in_channels, eps=eps, elementwise_affine=True, bias=True)
            self.conv_shortcut = conv_op(
                in_channels=in_channels, out_channels=out_channels,
                kernel_size=1, stride=1, is_causal=is_causal,
            )

        # Noise injection (unused for inference, kept for compat)
        self.per_channel_scale1 = None
        self.per_channel_scale2 = None
        if inject_noise:
            self.per_channel_scale1 = nn.Parameter(torch.zeros(in_channels, 1, 1))
            self.per_channel_scale2 = nn.Parameter(torch.zeros(in_channels, 1, 1))

        # Timestep conditioning (unused for inference, kept for compat)
        self.scale_shift_table = None
        if timestep_conditioning:
            self.scale_shift_table = nn.Parameter(torch.randn(4, in_channels) / in_channels ** 0.5)

    def forward(
        self, hidden_states: torch.Tensor,
        temb: Optional[torch.Tensor] = None,
        generator: Optional[torch.Generator] = None,
    ) -> torch.Tensor:
        inputs = hidden_states

        hidden_states = self.norm1(hidden_states.permute(0, 2, 3, 4, 1)).permute(0, 4, 1, 2, 3)

        if self.scale_shift_table is not None and temb is not None:
            temb = temb.unflatten(1, (4, -1)) + self.scale_shift_table[None, ..., None, None, None]
            shift_1, scale_1, shift_2, scale_2 = temb.unbind(dim=1)
            hidden_states = hidden_states * (1 + scale_1) + shift_1

        if self.is_upsampler_modified:
            hidden_states = self.replace_nonlinearity(hidden_states)
        else:
            hidden_states = self.nonlinearity(hidden_states)

        hidden_states = self.conv1(hidden_states)

        if self.per_channel_scale1 is not None and generator is not None:
            spatial_shape = hidden_states.shape[-2:]
            spatial_noise = torch.randn(spatial_shape, generator=generator, device=hidden_states.device, dtype=hidden_states.dtype)[None]
            hidden_states = hidden_states + (spatial_noise * self.per_channel_scale1)[None, :, None, ...]

        hidden_states = self.norm2(hidden_states.permute(0, 2, 3, 4, 1)).permute(0, 4, 1, 2, 3)

        if self.scale_shift_table is not None and temb is not None:
            hidden_states = hidden_states * (1 + scale_2) + shift_2

        hidden_states = self.nonlinearity(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.conv2(hidden_states)

        if self.per_channel_scale2 is not None and generator is not None:
            spatial_shape = hidden_states.shape[-2:]
            spatial_noise = torch.randn(spatial_shape, generator=generator, device=hidden_states.device, dtype=hidden_states.dtype)[None]
            hidden_states = hidden_states + (spatial_noise * self.per_channel_scale2)[None, :, None, ...]

        if self.norm3 is not None:
            inputs = self.norm3(inputs.permute(0, 2, 3, 4, 1)).permute(0, 4, 1, 2, 3)
        if self.conv_shortcut is not None:
            inputs = self.conv_shortcut(inputs)

        hidden_states = hidden_states + inputs
        return hidden_states


# ============================================================
# TurboVAEDMidBlock3d
# ============================================================

class TurboVAEDMidBlock3d(nn.Module):
    """Middle block with multiple ResNet layers."""

    def __init__(
        self,
        in_channels: int,
        num_layers: int = 1,
        dropout: float = 0.0,
        resnet_eps: float = 1e-6,
        resnet_act_fn: str = "swish",
        is_causal: bool = True,
        inject_noise: bool = False,
        timestep_conditioning: bool = False,
        is_dw_conv: bool = False,
        dw_kernel_size: int = 3,
    ):
        super().__init__()

        resnets = []
        for _ in range(num_layers):
            resnets.append(
                TurboVAEDResnetBlock3d(
                    in_channels=in_channels,
                    out_channels=in_channels,
                    dropout=dropout,
                    eps=resnet_eps,
                    non_linearity=resnet_act_fn,
                    is_causal=is_causal,
                    inject_noise=inject_noise,
                    timestep_conditioning=timestep_conditioning,
                    is_dw_conv=is_dw_conv,
                    dw_kernel_size=dw_kernel_size,
                )
            )
        self.resnets = nn.ModuleList(resnets)

    def forward(
        self, hidden_states: torch.Tensor,
        temb: Optional[torch.Tensor] = None,
        generator: Optional[torch.Generator] = None,
    ) -> torch.Tensor:
        for resnet in self.resnets:
            hidden_states = resnet(hidden_states, temb, generator)
        return hidden_states


# ============================================================
# TurboVAEDUpBlock3d
# ============================================================

class TurboVAEDUpBlock3d(nn.Module):
    """Up block with optional spatio-temporal upsampling and ResNet layers."""

    def __init__(
        self,
        in_channels: int,
        out_channels: Optional[int] = None,
        num_layers: int = 1,
        dropout: float = 0.0,
        resnet_eps: float = 1e-6,
        resnet_act_fn: str = "swish",
        spatio_temporal_scale: bool = True,
        is_causal: bool = True,
        inject_noise: bool = False,
        timestep_conditioning: bool = False,
        upsample_residual: bool = False,
        upscale_factor: int = 1,
        is_dw_conv: bool = False,
        dw_kernel_size: int = 3,
        spatio_only: bool = False,
        is_video_dc_ae: bool = False,
    ):
        super().__init__()

        out_channels = out_channels or in_channels

        self.conv_in = None
        if in_channels != out_channels:
            self.conv_in = TurboVAEDResnetBlock3d(
                in_channels=in_channels,
                out_channels=out_channels,
                dropout=dropout,
                eps=resnet_eps,
                non_linearity=resnet_act_fn,
                is_causal=is_causal,
                inject_noise=inject_noise,
                timestep_conditioning=timestep_conditioning,
                is_dw_conv=is_dw_conv,
                dw_kernel_size=dw_kernel_size,
            )

        self.upsamplers = None
        if spatio_temporal_scale:
            stride_up = (2, 2, 2) if not spatio_only else (1, 2, 2)
            self.upsamplers = nn.ModuleList([
                TurboVAEDUpsampler3d(
                    out_channels * upscale_factor,
                    stride=stride_up,
                    is_causal=is_causal,
                    residual=upsample_residual,
                    upscale_factor=upscale_factor,
                    is_video_dc_ae=is_video_dc_ae,
                )
            ])

        resnets = []
        for _ in range(num_layers):
            resnets.append(
                TurboVAEDResnetBlock3d(
                    in_channels=out_channels,
                    out_channels=out_channels,
                    dropout=dropout,
                    eps=resnet_eps,
                    non_linearity=resnet_act_fn,
                    is_causal=is_causal,
                    inject_noise=inject_noise,
                    timestep_conditioning=timestep_conditioning,
                    is_dw_conv=is_dw_conv,
                    dw_kernel_size=dw_kernel_size,
                    is_upsampler_modified=spatio_temporal_scale,
                )
            )
        self.resnets = nn.ModuleList(resnets)

    def forward(
        self, hidden_states: torch.Tensor,
        temb: Optional[torch.Tensor] = None,
        generator: Optional[torch.Generator] = None,
    ) -> torch.Tensor:
        if self.conv_in is not None:
            hidden_states = self.conv_in(hidden_states, temb, generator)
        if self.upsamplers is not None:
            for upsampler in self.upsamplers:
                hidden_states = upsampler(hidden_states)
        for resnet in self.resnets:
            hidden_states = resnet(hidden_states, temb, generator)
        return hidden_states


# ============================================================
# TurboVAEDDecoder3d
# ============================================================

class TurboVAEDDecoder3d(nn.Module):
    """
    The Turbo-VAED decoder.

    Args are designed to be compatible with the config dict from the
    AutoencoderKLTurboVAED class.  The `build_turbo_vaed_decoder()` factory
    performs the key name mapping from the diffusers-style config.
    """

    def __init__(
        self,
        in_channels: int = 128,
        out_channels: int = 3,
        block_out_channels: Tuple[int, ...] = (128, 256, 512, 512),
        spatio_temporal_scaling: Tuple[bool, ...] = (True, True, True, False),
        layers_per_block: Tuple[int, ...] = (4, 3, 3, 3, 4),
        patch_size: int = 4,
        patch_size_t: int = 1,
        resnet_norm_eps: float = 1e-6,
        is_causal: bool = False,
        inject_noise: Tuple[bool, ...] = (False, False, False, False, False),
        timestep_conditioning: bool = False,
        upsample_residual: Tuple[bool, ...] = (False, False, False, False),
        upsample_factor: Tuple[int, ...] = (1, 1, 1, 1),
        decoder_is_dw_conv: Tuple[bool, ...] = (False, False, False, False, False),
        decoder_dw_kernel_size: int = 3,
        spatio_only: Tuple[bool, ...] = (False, False, False, False),
        upsampling: bool = False,
        is_video_dc_ae: bool = False,
    ):
        super().__init__()

        self.patch_size = patch_size
        self.patch_size_t = patch_size_t
        self.out_channels = out_channels
        self.upsampling = upsampling

        # Reverse for decoder (built bottom-up)
        block_out_channels = tuple(reversed(block_out_channels))
        spatio_temporal_scaling = tuple(reversed(spatio_temporal_scaling))
        layers_per_block = tuple(reversed(layers_per_block))
        inject_noise = tuple(reversed(inject_noise))
        upsample_residual = tuple(reversed(upsample_residual))
        upsample_factor = tuple(reversed(upsample_factor))
        decoder_is_dw_conv = tuple(reversed(decoder_is_dw_conv))
        spatio_only = tuple(reversed(spatio_only))

        output_channel = block_out_channels[0]

        # Conv in
        self.conv_in = TurboVAEDCausalConv3d(
            in_channels=in_channels, out_channels=output_channel, kernel_size=3, stride=1, is_causal=is_causal,
        )

        # Mid block
        self.mid_block = TurboVAEDMidBlock3d(
            in_channels=output_channel,
            num_layers=layers_per_block[0],
            resnet_eps=resnet_norm_eps,
            is_causal=is_causal,
            inject_noise=inject_noise[0],
            timestep_conditioning=timestep_conditioning,
            is_dw_conv=decoder_is_dw_conv[0],
            dw_kernel_size=decoder_dw_kernel_size,
        )

        # Up blocks
        num_block_out_channels = len(block_out_channels)
        self.up_blocks = nn.ModuleList([])
        for i in range(num_block_out_channels):
            input_channel = output_channel // upsample_factor[i]
            output_channel = block_out_channels[i] // upsample_factor[i]

            up_block = TurboVAEDUpBlock3d(
                in_channels=input_channel,
                out_channels=output_channel,
                num_layers=layers_per_block[i + 1],
                resnet_eps=resnet_norm_eps,
                spatio_temporal_scale=spatio_temporal_scaling[i],
                is_causal=is_causal,
                inject_noise=inject_noise[i + 1],
                timestep_conditioning=timestep_conditioning,
                upsample_residual=upsample_residual[i],
                upscale_factor=upsample_factor[i],
                is_dw_conv=decoder_is_dw_conv[i + 1],
                dw_kernel_size=decoder_dw_kernel_size,
                spatio_only=spatio_only[i],
                is_video_dc_ae=is_video_dc_ae,
            )
            self.up_blocks.append(up_block)

        # Final 2D upsamplers (pixel shuffle)
        if self.patch_size >= 2:
            self.norm_up_1 = RMSNorm(output_channel, eps=1e-8, elementwise_affine=False)
            self.upsampler2d_1 = TurboVAEDConv2dSplitUpsampler(
                in_channels=output_channel, kernel_size=3, stride=2,
            )
            output_channel = output_channel // (2 * 2)

        if self.patch_size >= 4:
            self.norm_up_2 = RMSNorm(output_channel, eps=1e-8, elementwise_affine=False)
            self.upsampler2d_2 = TurboVAEDConv2dUpsampler(
                in_channels=output_channel, kernel_size=3, stride=2,
            )
            output_channel = output_channel // (2 * 2)

        # Out norm
        if self.patch_size == 1:
            self.norm_out = RMSNorm(output_channel, eps=1e-8, elementwise_affine=False)

        self.conv_act = nn.SiLU()
        self.conv_out = TurboVAEDCausalConv3d(
            in_channels=output_channel, out_channels=self.out_channels, kernel_size=3, stride=1, is_causal=is_causal,
        )

    def forward(
        self, hidden_states: torch.Tensor,
        temb: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        # hidden_states: [B, C, T, H, W]
        hidden_states = self.conv_in(hidden_states)

        hidden_states = self.mid_block(hidden_states, temb)

        for up_block in self.up_blocks:
            hidden_states = up_block(hidden_states, temb)

        # Final 2D spatial upsampling
        if self.patch_size >= 2:
            hidden_states = self.norm_up_1(hidden_states.permute(0, 2, 3, 4, 1)).permute(0, 4, 1, 2, 3)
            hidden_states = self.conv_act(hidden_states)
            hidden_states_array = []
            for t in range(hidden_states.shape[2]):
                h = self.upsampler2d_1(hidden_states[:, :, t, :, :])
                hidden_states_array.append(h)
            hidden_states = torch.stack(hidden_states_array, dim=2)

        if self.patch_size >= 4:
            hidden_states = self.norm_up_2(hidden_states.permute(0, 2, 3, 4, 1)).permute(0, 4, 1, 2, 3)
            hidden_states = self.conv_act(hidden_states)
            hidden_states = self.upsampler2d_2(hidden_states)

        if self.patch_size == 1:
            hidden_states = self.norm_out(hidden_states.permute(0, 2, 3, 4, 1)).permute(0, 4, 1, 2, 3)
        else:
            variance = hidden_states.pow(2).mean(1, keepdim=True)
            hidden_states = hidden_states * torch.rsqrt(variance + 1e-8)

        hidden_states = self.conv_act(hidden_states)
        hidden_states = self.conv_out(hidden_states)
        return hidden_states


# ============================================================
# Factory: build decoder from AutoencoderKLTurboVAED config dict
# ============================================================

# Maps from AutoencoderKLTurboVAED config key → TurboVAEDDecoder3d arg
_DECODER_CONFIG_KEY_MAP = {
    "latent_channels": "in_channels",
    "out_channels": "out_channels",
    "decoder_block_out_channels": "block_out_channels",
    "decoder_layers_per_block": "layers_per_block",
    "patch_size": "patch_size",
    "patch_size_t": "patch_size_t",
    "resnet_norm_eps": "resnet_norm_eps",
    "decoder_causal": "is_causal",
    "timestep_conditioning": "timestep_conditioning",
    "decoder_inject_noise": "inject_noise",
    "upsample_residual": "upsample_residual",
    "upsample_factor": "upsample_factor",
    "decoder_is_dw_conv": "decoder_is_dw_conv",
    "decoder_dw_kernel_size": "decoder_dw_kernel_size",
    "decoder_spatio_only": "spatio_only",
    "is_video_dc_ae": "is_video_dc_ae",
}


def build_turbo_vaed_decoder(config: dict) -> TurboVAEDDecoder3d:
    """
    Build a TurboVAEDDecoder3d from an AutoencoderKLTurboVAED config dict
    (as found in the config JSON files at
    https://github.com/hustvl/Turbo-VAED/tree/main/configs).

    The config dict contains keys like `latent_channels`, `decoder_block_out_channels`,
    etc. which are mapped to the TurboVAEDDecoder3d constructor arguments.

    Args:
        config: Config dictionary from a Turbo-VAED JSON config file.

    Returns:
        A TurboVAEDDecoder3d instance (weights NOT loaded).
    """
    decoder_kwargs = {}

    for config_key, decoder_key in _DECODER_CONFIG_KEY_MAP.items():
        if config_key in config:
            decoder_kwargs[decoder_key] = config[config_key]

    # Handle spatio_temporal_scaling: use decoder_spatio_temporal_scaling if present,
    # otherwise fall back to spatio_temporal_scaling
    if "decoder_spatio_temporal_scaling" in config:
        decoder_kwargs["spatio_temporal_scaling"] = config["decoder_spatio_temporal_scaling"]
    elif "spatio_temporal_scaling" in config:
        decoder_kwargs["spatio_temporal_scaling"] = config["spatio_temporal_scaling"]

    # Set defaults for keys that might be missing
    decoder_kwargs.setdefault("in_channels", 128)
    decoder_kwargs.setdefault("out_channels", 3)
    decoder_kwargs.setdefault("block_out_channels", (128, 256, 512, 512))
    decoder_kwargs.setdefault("spatio_temporal_scaling", (True, True, True, False))
    decoder_kwargs.setdefault("layers_per_block", (4, 3, 3, 3, 4))
    decoder_kwargs.setdefault("patch_size", 4)
    decoder_kwargs.setdefault("is_causal", False)
    decoder_kwargs.setdefault("timestep_conditioning", False)
    decoder_kwargs.setdefault("decoder_is_dw_conv", (False, False, False, False, False))
    decoder_kwargs.setdefault("decoder_dw_kernel_size", 3)
    decoder_kwargs.setdefault("resnet_norm_eps", 1e-6)
    decoder_kwargs.setdefault("inject_noise", (False, False, False, False, False))
    decoder_kwargs.setdefault("upsample_residual", (False, False, False, False))
    decoder_kwargs.setdefault("upsample_factor", (1, 1, 1, 1))
    decoder_kwargs.setdefault("spatio_only", (False, False, False, False))
    decoder_kwargs.setdefault("is_video_dc_ae", False)

    return TurboVAEDDecoder3d(**decoder_kwargs)


In [8]:
# @title 7. Setup Python Path
import sys
sys.path.insert(0, CODE_DIR)
os.environ["MOBILEI2V_CACHE_DIR"] = CACHE_DIR
os.environ["MOBILEI2V_CODE_DIR"] = CODE_DIR

import importlib
import model_utils
importlib.reload(model_utils)
print(f"model_utils.MODEL_CACHE_DIR = {model_utils.MODEL_CACHE_DIR}")

model_utils.MODEL_CACHE_DIR = C:\model\cache


In [9]:
# @title 8. Convert VAE Encoder
# Downloads LTX-Video VAE, exports encoder to ONNX.

import logging, time
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

device = model_utils.get_device()
torch_dtype = model_utils.get_torch_dtype(device)
model_utils.enable_cuda_optimizations(device)

from diffusers import AutoencoderKLLTXVideo
from pathlib import Path

# Load VAE
vae = AutoencoderKLLTXVideo.from_pretrained(
    "Lightricks/LTX-Video",
    subfolder="vae",
    torch_dtype=torch_dtype,
    cache_dir=CACHE_DIR,
)
vae.to(device)
vae.eval()
print(f"VAE loaded: {sum(p.numel() for p in vae.parameters())/1e6:.2f}M params")

# Wrapper - handles 4D->5D->4D conversion
class VAEEncoderWrapper(torch.nn.Module):
    def __init__(self, vae):
        super().__init__()
        self.vae = vae
    def forward(self, pixel_values):
        x = pixel_values.unsqueeze(2)
        dist = self.vae.encode(x)
        latents = dist.latent_dist.sample()
        latents = latents * self.vae.config.scaling_factor
        return latents.squeeze(2)

wrapper = VAEEncoderWrapper(vae).to(device)
wrapper.eval()

dummy = torch.randn(1, 3, 720, 1280, dtype=torch_dtype, device=device)
dummy_inputs = {"pixel_values": dummy}

onnx_path = model_utils.export_onnx(
    model=wrapper,
    model_name="vae_encoder",
    output_dir=MODEL_DIR,
    dummy_inputs=dummy_inputs,
    dynamic_axes={
        "pixel_values": {0: "batch", 2: "height", 3: "width"},
        "latent":       {0: "batch", 2: "latent_height", 3: "latent_width"},
    },
    input_names=["pixel_values"],
    output_names=["latent"],
    verbose=False,
)

# Verify
feeds = {"pixel_values": dummy.cpu().numpy()}
model_utils.verify_onnx(onnx_path, feeds, ["latent"])
print(f"VAE Encoder saved to: {onnx_path}")


INFO Using CUDA device: NVIDIA GeForce GTX 1050 Ti
INFO CUDA optimizations enabled: cudnn.benchmark, float32_matmul_precision=high
C:\Users\himad\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\himad\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
INFO HTTP Request: HEAD https://huggingface.co/Lightricks/LTX-Video/resolve/main/vae/config.json "HTTP/1.1 307 Temporary Redirect"
WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO HTTP Request: HEAD https://huggingfa

VAE loaded: 419.19M params


INFO Export complete — verifying ONNX model ...
INFO Repaired 1 Transpose node(s) with -1 perm
INFO ONNX check passed for \content\mobilei2v_onnx\vae_encoder.onnx
INFO vae_encoder.onnx size: 344.34 MB
INFO Verifying \content\mobilei2v_onnx\vae_encoder.onnx with onnxruntime ...
INFO ONNX Runtime using provider: CUDAExecutionProvider
INFO Verification passed — 1 output(s) produced with shapes: [(1, 128, 23, 40)]


VAE Encoder saved to: \content\mobilei2v_onnx\vae_encoder.onnx


In [10]:
import logging, time
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

device = model_utils.get_device()
torch_dtype = model_utils.get_torch_dtype(device)
model_utils.enable_cuda_optimizations(device)

# @title 9. Convert Qwen2 Text Encoder
# Downloads Qwen2-0.5B, exports text encoder to ONNX.

from transformers import AutoModel, AutoTokenizer
import numpy as np

model_id = "Qwen/Qwen2-0.5B"
print(f"Loading {model_id} ...")
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=CACHE_DIR)
text_model = AutoModel.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    attn_implementation="sdpa",
    cache_dir=CACHE_DIR,
)
text_model.to(device)
text_model.eval()
print(f"Qwen2 loaded: {sum(p.numel() for p in text_model.parameters())/1e6:.2f}M params")

# Wrapper - strips LM head, returns last_hidden_state
class Qwen2EncoderWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=False,
            return_dict=True,
        )
        return outputs.last_hidden_state

wrapper = Qwen2EncoderWrapper(text_model).to(device)
wrapper.eval()

# Dummy inputs
max_length = 300
dummy_ids = torch.randint(0, 1000, (1, max_length), device=device, dtype=torch.long)
dummy_mask = torch.ones(1, max_length, device=device, dtype=torch.long)
dummy_inputs = {"input_ids": dummy_ids, "attention_mask": dummy_mask}

onnx_path = model_utils.export_onnx(
    model=wrapper,
    model_name="qwen2_encoder",
    output_dir=MODEL_DIR,
    dummy_inputs=dummy_inputs,
    dynamic_axes={
        "input_ids":        {0: "batch", 1: "sequence"},
        "attention_mask":   {0: "batch", 1: "sequence"},
        "last_hidden_state": {0: "batch", 1: "sequence"},
    },
    input_names=["input_ids", "attention_mask"],
    output_names=["last_hidden_state"],
    verbose=False,
)

# Verify
feeds = {"input_ids": dummy_ids.cpu().numpy(), "attention_mask": dummy_mask.cpu().numpy()}
model_utils.verify_onnx(onnx_path, feeds, ["last_hidden_state"])
print(f"Qwen2 Encoder saved to: {onnx_path}")

INFO Using CUDA device: NVIDIA GeForce GTX 1050 Ti
INFO CUDA optimizations enabled: cudnn.benchmark, float32_matmul_precision=high


Loading Qwen/Qwen2-0.5B ...


INFO HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-0.5B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-0.5B/91d2aff3f957f99e4c74c962f2f408dcc88a18d8/config.json "HTTP/1.1 200 OK"
INFO HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-0.5B/91d2aff3f957f99e4c74c962f2f408dcc88a18d8/config.json "HTTP/1.1 200 OK"
C:\Users\himad\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\content\model_cache\models--Qwen--Qwen2-0.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_h

Qwen2 loaded: 494.03M params


C:\Users\himad\AppData\Roaming\Python\Python312\site-packages\transformers\masking_utils.py:208: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
C:\Users\himad\AppData\Roaming\Python\Python312\site-packages\transformers\masking_utils.py:277: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if padding_mask is not None and padding_mask.shape[-1] > kv_length:
C:\Users\himad\AppData\Roaming\Python\Python312\site-packages\transformers\cache_utils.py:132: TracerWarning: torch.tensor resu

RuntimeError: Expected output 'last_hidden_hidden_state' not found in model. Available: ['last_hidden_state']

In [11]:
import gc

# Delete large objects to free up memory
# These objects were created in previous cells.
# Use a try-except block in case some objects are not yet defined
try: del vae
except NameError: pass
try: del text_model
except NameError: pass
try: del model
except NameError: pass
try: del wrapper
except NameError: pass
try: del checkpoint
except NameError: pass
try: del filtered
except NameError: pass
try: del dummy
except NameError: pass
try: del dummy_ids
except NameError: pass
try: del dummy_mask
except NameError: pass
try: del dummy_inputs
except NameError: pass
try: del latent
except NameError: pass
try: del text_emb
except NameError: pass
try: del timestep
except NameError: pass
try: del feeds
except NameError: pass

# Force garbage collection
gc.collect()

# Clear CUDA cache
torch.cuda.empty_cache()
torch.cuda.memory.empty_cache()
print("GPU memory released.")

GPU memory released.


In [12]:
import logging, time
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

device = model_utils.get_device()
torch_dtype = model_utils.get_torch_dtype(device)
model_utils.enable_cuda_optimizations(device)

import _pickle
from huggingface_hub import hf_hub_download

# Dynamically import mobiledit after writing it to file
import sys
sys.path.insert(0, '/content/mobilei2v_code')
from models.mobiledit import MobileditONNXWrapper, mobiledit_300m_P1_D16

# Download checkpoint
checkpoint_path = hf_hub_download(
    repo_id="hustvl/MobileI2V",
    filename="hybrid_371.pth",
    local_dir=CACHE_DIR,
    local_dir_use_symlinks=False,
)
print(f"Checkpoint: {checkpoint_path}")

# Create model
model = mobiledit_300m_P1_D16(attn_type='flash').to(device=device, dtype=torch_dtype)
print(f"Model created: {sum(p.numel() for p in model.parameters())/1e6:.2f}M params")

# Load checkpoint (filtering to only model keys)
try:
    # First attempt: load to CPU with weights_only=True to get state_dict directly
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
except _pickle.UnpicklingError:
    print("weights_only=True failed. Attempting to load with weights_only=False to CPU.")
    # Fallback: load full object to CPU if weights_only=True fails
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

ckpt_state = checkpoint.get("state_dict", checkpoint)
model_state = model.state_dict()
filtered = {}
skipped_pos = False
for key, value in ckpt_state.items():
    if key not in model_state:
        continue
    if key == "pos_embed" and value.shape != model_state[key].shape:
        print(f"pos_embed shape mismatch: checkpoint {list(value.shape)} vs model {list(model_state[key].shape)} - skipping")
        skipped_pos = True
        continue
    # Move to target device and dtype here
    filtered[key] = value.to(device=device, dtype=torch_dtype)

print(f"Loaded {len(filtered)} / {len(model_state)} keys")
model.load_state_dict(filtered, strict=False)
model.eval()

# Export
wrapper = MobileditONNXWrapper(model).to(device=device, dtype=torch_dtype)
wrapper.eval()

# Reduced batch size to 1 and temporal frames to 4 to avoid OOM during export
# and enable dynamic temporal axis
latent = torch.randn(1, 128, 4, 32, 32, device=device, dtype=torch_dtype)
text_emb = torch.randn(1, 300, 896, device=device, dtype=torch_dtype)
timestep = torch.randint(0, 1000, (1,), device=device, dtype=torch.long)

dummy_inputs = {"latent": latent, "text_emb": text_emb, "timestep": timestep}

onnx_path = model_utils.export_onnx(
    model=wrapper, model_name="mobilei2v_unet", output_dir=MODEL_DIR,
    dummy_inputs=dummy_inputs,
    dynamic_axes={
        "latent": {0: "batch", 2: "temporal_frames", 3: "height", 4: "width"},
        "text_emb": {0: "batch", 1: "text_tokens"},
        "timestep": {0: "batch"},
        "denoised_latent": {0: "batch", 2: "temporal_frames", 3: "height", 4: "width"},
    },
    input_names=["latent", "text_emb", "timestep"],
    output_names=["denoised_latent"],
    verbose=True,
)

# Verify
feeds = {
    "latent": latent.cpu().numpy(),
    "text_emb": text_emb.cpu().numpy(),
    "timestep": timestep.cpu().numpy(),
}
model_utils.verify_onnx(onnx_path, feeds, ["denoised_latent"])
print(f"MobileI2V UNet saved to: {onnx_path}")

INFO Using CUDA device: NVIDIA GeForce GTX 1050 Ti
INFO CUDA optimizations enabled: cudnn.benchmark, float32_matmul_precision=high
INFO HTTP Request: HEAD https://huggingface.co/hustvl/MobileI2V/resolve/main/hybrid_371.pth "HTTP/1.1 302 Found"


Checkpoint: C:\content\model_cache\hybrid_371.pth
Model created: 276.93M params
weights_only=True failed. Attempting to load with weights_only=False to CPU.


INFO Exporting mobilei2v_unet -> \content\mobilei2v_onnx\mobilei2v_unet.onnx
WARNING dynamo_export failed for 'mobilei2v_unet' — falling back to legacy torch.onnx.export
Traceback (most recent call last):
  File "c:\content/mobilei2v_code\model_utils.py", line 236, in export_onnx
    export_options = torch.onnx.ExportOptions(
                     ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: module 'torch.onnx' has no attribute 'ExportOptions'


pos_embed shape mismatch: checkpoint [1, 920, 1152] vs model [1, 1024, 1152] - skipping
Loaded 89 / 168 keys


c:\content/mobilei2v_code\models\mobiledit.py:1214: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if T % self.patch_size != 0:
c:\content/mobilei2v_code\models\mobiledit.py:1216: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if H % self.patch_size != 0:
c:\content/mobilei2v_code\models\mobiledit.py:1218: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to o

InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Invalid input name: text_emb

In [ ]:
# @title 11. Convert Turbo-VAED Decoder (vendored model)
# Downloads config from GitHub + checkpoint from HuggingFace, builds decoder
# via vendored turbo_vaed_model.py, exports to ONNX.
#
# Input:  [B, C, T, H, W]   (C=128 for LTX variant)
# Output: [B, 3, T*8, H*32, W*32]

import json
import urllib.request
import os

from models.turbo_vaed_model import build_turbo_vaed_decoder
from huggingface_hub import hf_hub_download

# --- Configuration ---
VARIANT = "LTX"  # Options: LTX, CogVideo5B, HunyuanVideo
VAED_HF_REPO = "hustvl/Turbo-VAED"

VARIANT_CHANNELS = {"LTX": 128, "CogVideo5B": 16, "HunyuanVideo": 16}
latent_channels = VARIANT_CHANNELS[VARIANT]

config_url = (
    "https://raw.githubusercontent.com/hustvl/Turbo-VAED/main/configs/"
    f"Turbo-VAED-{VARIANT}.json"
)

# --- Step 1: Download config from GitHub ---
print(f"Downloading config from {config_url} ...")
with urllib.request.urlopen(config_url) as resp:
    config = json.loads(resp.read().decode())
print(f"Config loaded: {len(config)} keys")

# --- Step 2: Download checkpoint from HuggingFace ---
vae_cache = os.path.join(CACHE_DIR, f"turbo_vaed_{VARIANT}")
os.makedirs(vae_cache, exist_ok=True)

ckpt_path = hf_hub_download(
    repo_id=VAED_HF_REPO,
    filename=f"Turbo-VAED-{VARIANT}.pth",
    local_dir=vae_cache,
    resume_download=True,
)
print(f"Checkpoint: {ckpt_path}")

# --- Step 3: Build decoder ---
print("Building Turbo-VAED decoder (vendored model) ...")
decoder = build_turbo_vaed_decoder(config)
n_params = sum(p.numel() for p in decoder.parameters()) / 1e6
print(f"Decoder built: {n_params:.2f}M params")

# --- Step 4: Load state dict (strip "decoder." prefix) ---
print("Loading checkpoint ...")
state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=True)

decoder_keys = {}
for k, v in state_dict.items():
    if k.startswith("decoder."):
        decoder_keys[k[len("decoder."):]] = v
if not decoder_keys:
    decoder_keys = state_dict

missing, unexpected = decoder.load_state_dict(decoder_keys, strict=False)
if missing:
    print(f"  Missing keys: {len(missing)}")
if unexpected:
    print(f"  Non-decoder keys skipped: {len(unexpected)}")

decoder.to(device, dtype=torch_dtype)
decoder.eval()
n_params = sum(p.numel() for p in decoder.parameters()) / 1e6
print(f"Decoder ready: {n_params:.2f}M params")

# --- Step 5: Wrap ---
class TurbVAEDWrapper(torch.nn.Module):
    def __init__(self, dec):
        super().__init__()
        self.decoder = dec
    def forward(self, latent):
        return self.decoder(latent)

wrapper = TurbVAEDWrapper(decoder).to(device, dtype=torch_dtype)
wrapper.eval()

# --- Step 6: Build 5D dummy input ---
dummy_latent = torch.randn(
    1, latent_channels, 5, 23, 40,
    dtype=torch_dtype, device=device,
)
dummy_inputs = {"latent": dummy_latent}
print(f"Dummy input shape: {list(dummy_latent.shape)}")

# --- Step 7: Export to ONNX ---
onnx_path = model_utils.export_onnx(
    model=wrapper,
    model_name="turbo_vaed",
    output_dir=MODEL_DIR,
    dummy_inputs=dummy_inputs,
    dynamic_axes={
        "latent": {0: "batch", 2: "num_frames", 3: "height", 4: "width"},
        "frames": {0: "batch", 2: "num_frames", 3: "height", 4: "width"},
    },
    input_names=["latent"],
    output_names=["frames"],
    verbose=False,
)

# --- Step 8: Verify ---
feeds = {"latent": dummy_latent.cpu().numpy()}
model_utils.verify_onnx(onnx_path, feeds, ["frames"])
print(f"Turbo-VAED saved to: {onnx_path}")


INFO HTTP Request: HEAD https://huggingface.co/hustvl/MobileI2V/resolve/main/vae_decoder/config.json "HTTP/1.1 404 Not Found"


  Download issue for config.json: 404 Client Error. (Request ID: Root=1-6a456ac7-16dda8dd577bd51959cf2f11;ae35ff40-667c-444c-9ad3-faca015760d9)

Entry Not Found for url: https://huggingface.co/hustvl/MobileI2V/resolve/main/vae_decoder/config.json.


INFO HTTP Request: HEAD https://huggingface.co/hustvl/MobileI2V/resolve/main/vae_decoder/model.safetensors "HTTP/1.1 404 Not Found"


  Download issue for model.safetensors: 404 Client Error. (Request ID: Root=1-6a456ac7-696237ca304ebd8d6d08883d;828b75af-c75e-4ab3-ab21-15033fdedf17)

Entry Not Found for url: https://huggingface.co/hustvl/MobileI2V/resolve/main/vae_decoder/model.safetensors.
Contents: ['.cache', 'vae_decoder']


INFO HTTP Request: GET https://huggingface.co/api/models/hustvl/MobileI2V/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
WARNING No files matched patterns ['vae_decoder/*', '*.json'] in repo hustvl/MobileI2V
INFO Downloading 0 file(s) from hustvl/MobileI2V ...


Loading decoder from: \content\model_cache\mobilei2v_vae_decoder


OSError: Error no file named config.json found in directory \content\model_cache\mobilei2v_vae_decoder.

In [ ]:
# @title 12. Summary & Download
import os, hashlib

onnx_files = [f for f in os.listdir(MODEL_DIR) if f.endswith(".onnx")]
onnx_files.sort()
print(f"{'Model':<25} {'Size (MB)':<12} {'SHA256':<20}")
print("=" * 60)
for f in onnx_files:
    path = os.path.join(MODEL_DIR, f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    h = hashlib.sha256()
    with open(path, "rb") as fp:
        for chunk in iter(lambda: fp.read(65536), b""):
            h.update(chunk)
    sha = h.hexdigest()[:16]
    print(f"{f:<25} {size_mb:<12.2f} {sha:<20}")

print()
print(f"Total: {len(onnx_files)} ONNX models in {MODEL_DIR}")

# Zip for download
import shutil
shutil.make_archive("/content/mobilei2v_onnx", "zip", MODEL_DIR)
print()
print("Download: /content/mobilei2v_onnx.zip")

from google.colab import files
files.download("/content/mobilei2v_onnx.zip")
